# Paper Figure Pipeline

This notebook is the single entry point for manuscript and supporting figure generation. Each figure-producing code cell renders one named figure and exports the PDF/SVG/PNG triplet with `save_triplet(fig, name, str(FIGURES))`.

Data roots are centralized in the setup cell:

- `DATA = PROJECT_ROOT / "data"`
- `OUTPUTS = PROJECT_ROOT / "outputs"`
- `FIGURES = OUTPUTS / "figures"`

Primary data sources used here are `outputs/master_kpi_table.csv`, `outputs/main_baseline/case2/dispatch.csv.gz`, `data/weather/houston_wet_bulb_combined.csv`, ERCOT LMP files under `data/ercot/`, `data/price_grid.csv`, `data/it_load.csv`, `config/capex_grid_s5.yaml`, and `outputs/figures/value_decomp_case2.csv`.

- Current manuscript figure cells appear first in the compiled `main.pdf` order. When manuscript figure order, timing, or panel composition changes, update this notebook order and the `Figure` / `Panel` code comments in the same pass.
All generated figure files are written to `outputs/figures/`. The manuscript reads that same directory through `\graphicspath{{../../outputs/figures/}}`, so re-running this notebook updates the figures used by the PDF.


## Sci-Figure / SciFiGear Color Options

This notebook uses the `sci-figure` palette below. Filled charts should normally use the first `n` colors from `MAIN_FILLS`: one color = light blue; two colors = light blue + salmon/pink; three colors = add yellow/orange; four colors = add gray. Line and marker strokes should use the coordinated deeper colors from `MAIN_STROKES`.

### Main manuscript sequences

| Name | Order / role | Hex values |
|---|---|---|
| `MAIN_FILLS` | Filled bars, blocks, heatmap categories | `#BEE8FF`, `#FF7F7F`, `#FFD37F`, `#B9B9B9` |
| `MAIN_STROKES` | Lines, marker edges, annotations | `#224686`, `#A46A4D`, `#036868`, `#555555` |
| `BLOCK_EDGE` | Filled block separator | `#FFFFFF` |

### Palette entries

| Key | Hex | Recommended role |
|---|---:|---|
| `fill_blue` | `#BEE8FF` | Low-saturation fill; first/default filled category |
| `fill_salmon` | `#FF7F7F` | Low-saturation fill; second filled category / negative or worse direction |
| `fill_orange` | `#FFD37F` | Low-saturation fill; third filled category |
| `fill_gray` | `#B9B9B9` | Low-saturation fill; fourth filled category / neutral or baseline |
| `stroke_teal` | `#036868` | High-saturation line / marker edge / cooling-water accent |
| `stroke_navy` | `#224686` | High-saturation line / marker edge; first/default line color |
| `stroke_clay` | `#A46A4D` | High-saturation line / marker edge; warm companion to salmon/pink |
| `accent_teal` | `#29BD9A` | Mid-saturation accent / callout |
| `accent_sky` | `#63C0EF` | Mid-saturation reference accent |
| `accent_purple` | `#9B77CB` | Mid-saturation extra accent, used sparingly |
| `case0` | `#B9B9B9` | Legacy case alias: grid-only / baseline |
| `case1` | `#86DED4` | Legacy case alias: BWRX no heat recovery |
| `case2` | `#036868` | Legacy case alias: BWRX cogeneration / hero case |
| `case3` | `#224686` | Legacy case alias: BWRX + absorption |
| `case4` | `#A46A4D` | Legacy case alias: NGCC on-site |
| `premium_pos` | `#29BD9A` | Legacy positive Premium alias |
| `premium_neg` | `#FF7F7F` | Legacy negative Premium alias |
| `ref` | `#63C0EF` | Legacy reference-marker alias |
| `comp_capital` | `#BEE8FF` | Legacy stacked-cost alias: capital |
| `comp_om` | `#FFD37F` | Legacy stacked-cost alias: O&M |
| `comp_fuel` | `#FF7F7F` | Legacy stacked-cost alias: fuel |
| `comp_grid_buy` | `#A46A4D` | Legacy stacked-cost alias: grid import |
| `comp_grid_sell` | `#62AE9E` | Legacy stacked-cost alias: grid export |


In [ ]:
SCIENTIFIC_FIGURE_COLOR_OPTIONS = {
    "sci_figure_main_fills": [
        "#BEE8FF",  # fill_blue
        "#FF7F7F",  # fill_salmon
        "#FFD37F",  # fill_orange
        "#B9B9B9",  # fill_gray
    ],
    "sci_figure_main_strokes": [
        "#224686",  # stroke_navy
        "#A46A4D",  # stroke_clay
        "#036868",  # stroke_teal
        "#555555",  # dark gray companion
    ],
    "sci_figure_accents_and_edges": [
        "#FFFFFF",  # block edge / white separator
        "#29BD9A",  # accent_teal
        "#63C0EF",  # accent_sky
        "#9B77CB",  # accent_purple
        "#86DED4",  # Nuclear-DC case1 mint teal
        "#62AE9E",  # grid-sell muted teal
    ],
    "nature_default_palette": [
        "#0F4D92",  # blue_main
        "#3775BA",  # blue_secondary
        "#DDF3DE",  # green_1
        "#AADCA9",  # green_2
        "#8BCF8B",  # green_3
        "#F6CFCB",  # red_1
        "#E9A6A1",  # red_2
        "#B64342",  # red_strong
        "#CFCECE",  # neutral_light
        "#767676",  # neutral_mid
        "#4D4D4D",  # neutral_dark
        "#272727",  # neutral_black
        "#FFD700",  # gold
        "#42949E",  # teal
        "#9A4D8E",  # violet
        "#EA84DD",  # magenta
    ],
    "nature_nmi_pastel": [
        "#484878",  # baseline_dark
        "#7884B4",  # baseline_mid
        "#B4C0E4",  # baseline_soft
        "#E4E4F0",  # ours_tiny
        "#E4CCD8",  # ours_base
        "#F0C0CC",  # ours_large
        "#E0E0F0",  # bg_lilac
        "#E0F0F0",  # bg_aqua
        "#F0E0D0",  # bg_peach
        "#D8D8D8",  # neutral_light
        "#A8A8A8",  # neutral_mid
        "#606060",  # neutral_dark
        "#2E9E44",  # delta_up
        "#E53935",  # delta_down
    ],
    "nature_imaging_dark_background": [
        "#000000",  # bg
        "#B8B8B8",  # context
        "#22D7E6",  # cyan
        "#FF2AD4",  # magenta
        "#FFFFFF",  # white
    ],
    "nature_material": [
        "#77D7D1",  # aqua
        "#33B5A5",  # teal
        "#B9A7E8",  # lilac
        "#7C6CCF",  # violet
        "#E53935",  # callout_red
        "#D9D9D9",  # neutral
    ],
    "nature_clinical": [
        "#272727",  # baseline
        "#E28E2C",  # week6
        "#D24B40",  # week13
        "#5B8FD6",  # week26
        "#7BAA5B",  # year1
        "#C45AD6",  # year2
        "#F2E6D9",  # group_band
    ],
    "nature_genomics": [
        "#D8D8D8",  # neutral_light
        "#8F8F8F",  # neutral_mid
        "#D9544D",  # wave1
        "#5B7FCA",  # wave2
        "#B89BD9",  # wave3
        "#4D4D4D",  # outline
    ],
}


In [ ]:
# Shared setup for all figure cells
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib.pyplot as plt
from IPython.display import display
import numpy as np
import pandas as pd
import yaml
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch, PathPatch
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from matplotlib.lines import Line2D

# sci-figure helpers (palette + apply_sci_style + save_triplet)
SCI_FIGURE_SCRIPTS = "/home/honglin/.claude/skills/sci-figure/scripts"
if SCI_FIGURE_SCRIPTS not in sys.path:
    sys.path.insert(0, SCI_FIGURE_SCRIPTS)
from sci_figure_helpers import (  # noqa: E402
    PALETTE,
    apply_sci_style,
    save_triplet,
    add_panel_label,
)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA = PROJECT_ROOT / "data"
OUTPUTS = PROJECT_ROOT / "outputs"
FIGURES = OUTPUTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

MANUSCRIPT_RC = {
    # Source lettering is intentionally a little larger than the finished
    # 7 pt artwork target because LaTeX scales each PDF to \linewidth.
    "font.size": 8.0,
    "axes.labelsize": 8.0,
    "xtick.labelsize": 8.0,
    "ytick.labelsize": 8.0,
    "legend.fontsize": 8.0,
}


def apply_manuscript_style(width: str = "ae_single") -> None:
    apply_sci_style(width)
    plt.rcParams.update(MANUSCRIPT_RC)


apply_manuscript_style("ae_single")  # final PDF lettering targets Elsevier 7 pt guidance

BODY_W = 7.0
SINGLE_W = BODY_W
DOUBLE_W = BODY_W
MAIN_FILLS = [
    PALETTE["fill_blue"],
    PALETTE["fill_salmon"],
    PALETTE["fill_orange"],
    PALETTE["fill_gray"],
]
MAIN_STROKES = [
    PALETTE["stroke_navy"],
    PALETTE["stroke_clay"],
    PALETTE["stroke_teal"],
    "#555555",
]
TAC_STACK_VOM_FILL = "#DDF3DE"
TAC_STACK_FUEL_FILL = PALETTE["fill_orange"]
TAC_STACK_C0_IMPORT_FILL = "#C1F4F0"
GRID_EXPORT_FILL = PALETTE["fill_gray"]
GRID_EXPORT_ALPHA = 1.0
BLOCK_EDGE = "#FFFFFF"
BLOCK_EDGE_LW = 0.5
EDGE_LW = BLOCK_EDGE_LW
TEXT_SIZE = MANUSCRIPT_RC["font.size"]
GRID_KW = dict(axis="y", alpha=0.3, linestyle="--", linewidth=0.5)
ZERO_LINE_KW = dict(color="#777777", linewidth=0.75, linestyle="--", alpha=0.9, zorder=10)
LINE_PLOT_LW = 1.85
THIN_LINE_PLOT_LW = 1.45
LINE_MARKER_SIZE = 6.2
LINE_MARKER_AREA = LINE_MARKER_SIZE ** 2
LINE_MARKER_EDGE_WIDTH = 1.0
BAR_LABEL_OFFSET_PT = 3
POINT_LABEL_OFFSET_PT = 5
VALUE_LABEL_OFFSET_PT = BAR_LABEL_OFFSET_PT
FIGURE2_LINE_COLORS = [MAIN_FILLS[1], MAIN_FILLS[0], MAIN_FILLS[3], MAIN_FILLS[2]]
FIGURE6_LINE_COLORS = [MAIN_FILLS[3], MAIN_FILLS[1], MAIN_FILLS[2]]
RENDER_UNUSED_FIGURES = False
BAR_GRADIENT_STEPS = 256


def annotate_vertical_value(ax, x, y, label, *, fontsize=TEXT_SIZE, offset_pt=VALUE_LABEL_OFFSET_PT,
                            color="#000000", force_positive_side=None, zorder=10):
    """Place a numeric label just outside a vertical bar or point endpoint."""
    sign_positive = y >= 0 if force_positive_side is None else bool(force_positive_side)
    dy = offset_pt if sign_positive else -offset_pt
    va = "bottom" if sign_positive else "top"
    ax.annotate(
        label,
        xy=(x, y), xycoords="data",
        xytext=(0, dy), textcoords="offset points",
        ha="center", va=va,
        fontsize=fontsize, color=color,
        zorder=zorder,
        annotation_clip=False,
    )


def annotate_horizontal_endpoint(ax, x, y, label, *, side, fontsize=TEXT_SIZE,
                                 offset_pt=POINT_LABEL_OFFSET_PT, color="#000000", zorder=10):
    """Place a numeric label just outside a horizontal range endpoint."""
    dx = -offset_pt if side == "left" else offset_pt
    ha = "right" if side == "left" else "left"
    ax.annotate(
        label,
        xy=(x, y), xycoords="data",
        xytext=(dx, 0), textcoords="offset points",
        ha=ha, va="center",
        fontsize=fontsize, color=color,
        zorder=zorder,
        annotation_clip=False,
    )


def maybe_render_unused_figure(render_fn) -> None:
    if RENDER_UNUSED_FIGURES:
        render_fn()


def _bar_gradient_cmap(color: str, steps: int = BAR_GRADIENT_STEPS) -> LinearSegmentedColormap:
    return LinearSegmentedColormap.from_list("white_to_bar_color", ["#FFFFFF", color], N=steps)


def apply_vertical_bar_gradients(ax, bars, colors, steps: int = BAR_GRADIENT_STEPS,
                                 *, white_at_zero: bool = False, gamma: float = 1.0) -> None:
    if isinstance(colors, str):
        colors = [colors] * len(bars)
    for bar, color in zip(bars, colors):
        height = bar.get_height()
        if abs(height) < 1e-12:
            bar.set_visible(False)
            continue
        xlim = ax.get_xlim()
        ylim = ax.get_ylim()
        x0 = bar.get_x()
        x1 = x0 + bar.get_width()
        y_start = bar.get_y()
        y_end = y_start + height
        y0, y1 = sorted((y_start, y_end))
        ramp = np.linspace(0.0, 1.0, steps) ** gamma
        if white_at_zero and height < 0:
            ramp = ramp[::-1]
        gradient = np.tile(ramp[:, None], (1, 2))
        ax.imshow(
            gradient,
            extent=(x0, x1, y0, y1),
            origin="lower",
            aspect="auto",
            cmap=_bar_gradient_cmap(color, steps),
            interpolation="bilinear",
            alpha=1.0,
            zorder=bar.get_zorder(),
        )
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        bar.set_visible(False)


def draw_horizontal_gradient_bar(ax, y, x_start, x_end, height, color, *, white_at="left",
                                 steps: int = BAR_GRADIENT_STEPS, gamma: float = 1.0) -> None:
    if x_end <= x_start:
        return
    ramp = np.linspace(0.0, 1.0, steps) ** gamma
    if white_at == "right":
        ramp = ramp[::-1]
    gradient = np.tile(ramp[None, :], (2, 1))
    ax.imshow(
        gradient,
        extent=(x_start, x_end, y - height / 2, y + height / 2),
        origin="lower",
        aspect="auto",
        cmap=_bar_gradient_cmap(color, steps),
        interpolation="nearest",
        resample=False,
        zorder=2,
    )


def fill_between_vertical_gradient(ax, x, y_lower, y_upper, color, *, alpha=0.32,
                                   steps: int = BAR_GRADIENT_STEPS, zorder=1.1) -> None:
    x = np.asarray(x, dtype=float)
    y_lower = np.asarray(y_lower, dtype=float)
    y_upper = np.asarray(y_upper, dtype=float)
    ymin = float(np.nanmin(np.minimum(y_lower, y_upper)))
    ymax = float(np.nanmax(np.maximum(y_lower, y_upper)))
    if not np.isfinite(ymin) or not np.isfinite(ymax) or ymax <= ymin:
        return
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    clip = ax.fill_between(x, y_lower, y_upper, facecolor="none", edgecolor="none",
                           linewidth=0, zorder=zorder)
    ramp = np.linspace(0.0, 1.0, steps)
    gradient = np.tile(ramp[:, None], (1, 2))
    im = ax.imshow(
        gradient,
        extent=(float(np.nanmin(x)), float(np.nanmax(x)), ymin, ymax),
        origin="lower",
        aspect="auto",
        cmap=_bar_gradient_cmap(color, steps),
        interpolation="bilinear",
        alpha=alpha,
        zorder=zorder,
    )
    paths = clip.get_paths()
    if paths:
        im.set_clip_path(PathPatch(paths[0], transform=ax.transData))
    clip.remove()
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)


CASE_LINE = {
    0: MAIN_STROKES[0],
    1: MAIN_STROKES[1],
    2: MAIN_STROKES[2],
    3: MAIN_STROKES[3],
}
CASE_FILL = {
    0: MAIN_FILLS[0],
    1: MAIN_FILLS[1],
    2: MAIN_FILLS[2],
    3: MAIN_FILLS[3],
}

In [ ]:
df = pd.read_csv(OUTPUTS / "master_kpi_table.csv")
print(f"Loaded {len(df)} rows from master_kpi_table.csv")
print(df.groupby("group").size())

# Convenience column for absolute Premium percentage
df["premium_pct"] = df["heat_recovery_premium"] * 100

# Case identity mapping for legends and colors
CASE_LABEL = {
    0: "Case 0 — Grid only",
    1: "Case 1 — Nuclear, no recovery",
    2: "Case 2 — Nuclear + cascade extraction",
    3: "Case 3 — NGCC on-site",
}
CASE_COLOR = {
    0: CASE_LINE[0],
    1: CASE_LINE[1],
    2: CASE_LINE[2],
    3: CASE_LINE[3],
}
CASE_MARKER = {0: "o", 1: "s", 2: "D", 3: "^"}

In [ ]:
# Figure 1: Conceptual mechanism for SMR-powered data-center cooling (fig1_system_schematic)
# Static/manual schematic used by main.tex; keep this marker first in the manuscript figure order.
fig1_path = FIGURES / "fig1_system_schematic.pdf"
assert fig1_path.exists(), f"Missing Figure 1 source file: {fig1_path}"
print(f"Figure 1 uses {fig1_path.relative_to(PROJECT_ROOT)}")


In [ ]:
# Figure 2: Hourly input traces driving the optimization (fig_inputs_overview)
# Panels A-D: wet-bulb distribution, ERCOT LMP duration curves, IT load, and absorption COP.
apply_manuscript_style("ae_double")  # full-width for the input-data figure

fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 5.0))


# (a) Houston wet-bulb seasonal box plot ---------------------------------
def figure_2_panel_a(ax):
    wb = pd.read_csv(DATA / "weather" / "houston_wet_bulb_combined.csv")
    wb["time"] = pd.to_datetime(wb["time"])
    wb["month"] = wb["time"].dt.month
    wb["year"] = wb["time"].dt.year
    wb23 = wb[wb["year"] == 2023]

    months = list(range(1, 13))
    data_by_month = [wb23[wb23.month == m].wet_bulb_C.dropna().values for m in months]
    bp = ax.boxplot(
        data_by_month,
        positions=months,
        widths=0.6,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color=FIGURE2_LINE_COLORS[2], linewidth=0.85),
        boxprops=dict(linewidth=0.5, edgecolor=BLOCK_EDGE),
        whiskerprops=dict(linewidth=0.5, color=FIGURE2_LINE_COLORS[2]),
        capprops=dict(linewidth=0.5, color=FIGURE2_LINE_COLORS[2]),
    )
    for patch in bp["boxes"]:
        patch.set_facecolor(PALETTE["fill_blue"])
        patch.set_alpha(0.7)

    # Mark the LiBr crystallization line (Twb + 5K approach >= 32C => Twb >= 27C)
    ax.axhline(27.0, color=FIGURE2_LINE_COLORS[0], linestyle="--", linewidth=LINE_PLOT_LW,
               label=r"LiBr gate: $T_{wb}\geq27\,^\circ$C")
    # Mark the COP design wet-bulb anchor
    ax.axhline(26.0, color=FIGURE2_LINE_COLORS[1], linestyle=":", linewidth=LINE_PLOT_LW,
               label=r"Design: $T_{wb}=26\,^\circ$C")

    ax.set_xticks(months)
    ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
    ax.set_ylabel(r"Wet-bulb $T_{wb}$ ($^\circ$C)")
    ax.set_xlabel("Month (2023)")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="lower center", bbox_to_anchor=(0.56, 1.02), ncol=2,
              frameon=True, facecolor="white", edgecolor="none", framealpha=0.86,
              fontsize=8, borderpad=0.25, handlelength=1.25,
              columnspacing=0.75,
              borderaxespad=0.0)
    # Figure 2, Panel A: Houston monthly wet-bulb distribution.
    add_panel_label(ax, "a")


# (b) ERCOT LMP duration curve -------------------------------------------
def figure_2_panel_b(ax):
    # Load raw ERCOT Houston Hub duration curves for each modeled year.
    # Do not synthesize missing years: the manuscript's numeric audit relies
    # on real hourly traces for all LMP-duration values.
    ercot_dir = DATA / "ercot"
    series_by_year = {}
    if ercot_dir.exists():
        for year in (2022, 2023, 2024):
            for candidate in [f"{year}_dam_lmp_houston.csv", f"ercot_dam_{year}.csv", f"dam_{year}.csv", f"{year}.csv"]:
                f = ercot_dir / candidate
                if f.exists():
                    df = pd.read_csv(f)
                    cols = [c for c in df.columns if "houston" in c.lower() or "lmp" in c.lower() or "price" in c.lower()]
                    if cols:
                        series_by_year[year] = df[cols[0]].dropna().values
                        break
    missing_years = [y for y in (2022, 2023, 2024) if y not in series_by_year]
    if missing_years:
        raise FileNotFoundError(f"Missing raw ERCOT LMP traces for years: {missing_years}")

    colors = {2022: FIGURE2_LINE_COLORS[0], 2023: FIGURE2_LINE_COLORS[1], 2024: FIGURE2_LINE_COLORS[2]}
    for year in (2022, 2023, 2024):
        s = np.sort(series_by_year[year])[::-1]
        x = np.arange(len(s)) / len(s) * 100.0
        ax.plot(x, s, color=colors[year], linewidth=LINE_PLOT_LW, label=str(year))

    ax.set_xlabel("Duration of year (\\%)")
    ax.set_ylabel(r"ERCOT LMP (\$/MWh$_\mathrm{e}$)")
    ax.set_ylim(bottom=-50)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="upper right", frameon=False, fontsize=8, title="ERCOT year")
    # Figure 2, Panel B: ERCOT LMP duration curves.
    add_panel_label(ax, "b")


# (c) IT load weekly profile ---------------------------------------------
def figure_2_panel_c(ax):
    df = pd.read_csv(DATA / "it_load.csv")
    # Pull a winter week and a summer week
    winter_start = 24 * 14
    summer_start = 24 * (31 + 28 + 31 + 30 + 31 + 30 + 14)  # mid-July
    hours = np.arange(168)
    winter = df.iloc[winter_start:winter_start + 168].IT_load_MW.values
    summer = df.iloc[summer_start:summer_start + 168].IT_load_MW.values
    winter_color = FIGURE2_LINE_COLORS[1]  # cool light blue for winter
    summer_color = FIGURE2_LINE_COLORS[0]  # warm salmon/pink for summer
    ax.plot(hours, winter, color=winter_color, linewidth=THIN_LINE_PLOT_LW,
            label="Winter (Jan 15--22)")
    ax.plot(hours, summer, color=summer_color, linewidth=THIN_LINE_PLOT_LW,
            label="Summer (Jul 15--22)")
    ax.axhline(df.IT_load_MW.mean(), color=FIGURE2_LINE_COLORS[2], linestyle=":", linewidth=LINE_PLOT_LW,
               label=fr"Annual mean {df.IT_load_MW.mean():.0f} MW$_\mathrm{{e}}$")
    ax.axhline(df.IT_load_MW.max(), color=FIGURE2_LINE_COLORS[3], linestyle="--", linewidth=LINE_PLOT_LW,
               label=fr"Annual peak {df.IT_load_MW.max():.0f} MW$_\mathrm{{e}}$")
    ax.set_xlabel("Hour of week")
    ax.set_ylabel(r"IT load $P_{IT}$ (MW$_\mathrm{e}$)")
    ax.set_xlim(0, 167)
    ax.set_ylim(0, 200)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    handles, labels = ax.get_legend_handles_labels()
    order = [0, 2, 1, 3]  # rows: Winter / Summer, then Mean / Peak
    ax.legend([handles[i] for i in order], [labels[i] for i in order],
              loc="upper center", bbox_to_anchor=(0.53, 0.98), ncol=2,
              frameon=True, facecolor="white", edgecolor="none", framealpha=0.86,
              fontsize=8, borderpad=0.25, columnspacing=0.75,
              handlelength=1.3, labelspacing=0.25, borderaxespad=0.0)
    # Figure 2, Panel C: data-center IT-load profile.
    add_panel_label(ax, "c")


# (d) Absorption COP(T_wb) -----------------------------------------------
def figure_2_panel_d(ax):
    Twb = np.linspace(10.0, 30.0, 256)
    # Reproduce builder.py _absorption_cop
    cop_nameplate = 1.30
    cop_houston_baseline = 1.10
    derate = 0.015
    baseline_T_wb = 26.0
    cop_lin = cop_houston_baseline - derate * (Twb - baseline_T_wb)
    cop_effective = np.maximum(np.minimum(cop_lin, cop_nameplate), 0.5)

    # Mask values above crystallization gate (cooling-water inlet >32C; Twb+5K>=32 -> Twb>=27)
    cop_gated = cop_effective.copy()
    cop_gated[Twb >= 27.0] = np.nan

    ax.plot(Twb, cop_effective, color=FIGURE2_LINE_COLORS[0], linewidth=LINE_PLOT_LW,
            linestyle="--", label="COP (without gate)")
    ax.plot(Twb, cop_gated, color=FIGURE2_LINE_COLORS[1], linewidth=LINE_PLOT_LW,
            label="COP (gated, used in model)")

    ax.axvspan(27.0, 30.0, alpha=0.15, color=PALETTE["fill_salmon"], zorder=0)
    ax.text(28.5, 1.18, "VCC\nbackup", ha="center", va="center", fontsize=8,
            color=FIGURE2_LINE_COLORS[0])

    ax.axhline(1.30, color=FIGURE2_LINE_COLORS[2], linestyle=":", linewidth=LINE_PLOT_LW)
    ax.text(11, 1.31, r"Nameplate cap 1.30", fontsize=8, color=FIGURE2_LINE_COLORS[2])
    ax.scatter([26.0], [1.10], s=42, marker="o",
               facecolor=FIGURE2_LINE_COLORS[3], edgecolor=BLOCK_EDGE, linewidth=LINE_MARKER_EDGE_WIDTH,
               zorder=4)
    ax.annotate("design anchor\n(26 $^\\circ$C, COP 1.10)",
                xy=(26.0, 1.10), xytext=(15, 0.88),
                arrowprops=dict(arrowstyle="-", color=FIGURE2_LINE_COLORS[3], linewidth=LINE_PLOT_LW),
                fontsize=8, ha="left", color="#222222")

    ax.set_xlabel(r"Wet-bulb $T_{wb}$ ($^\circ$C)")
    ax.set_ylabel(r"Absorption COP$_a$")
    ax.set_xlim(10, 30)
    ax.set_ylim(0.7, 1.4)
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.4)
    ax.legend(loc="lower left", frameon=False, fontsize=8)
    # Figure 2, Panel D: absorption-chiller COP curve.
    add_panel_label(ax, "d")


figure_2_panel_a(axes[0, 0])
figure_2_panel_b(axes[0, 1])
figure_2_panel_c(axes[1, 0])
figure_2_panel_d(axes[1, 1])

fig.tight_layout(h_pad=1.7, w_pad=1.2)
save_triplet(fig, "fig_inputs_overview", str(FIGURES))
display(fig)
plt.close(fig)

print("Saved fig_inputs_overview to outputs/figures/")
apply_manuscript_style("ae_single")  # restore the default style for the remaining cells


In [ ]:
# Figure 3: Total annualized cost decomposition (fig2_tac_stack)
# Single-panel manuscript figure.
def figure_3_tac_stack() -> None:
    base = df[df.group == "main_baseline"].sort_values("case_id").reset_index(drop=True)

    cases = base.case_id.astype(int).values
    labels = [f"C{c}" for c in cases]
    capex = base.capex_annual_usd.values / 1e6
    fom = base.fom_annual_usd.values / 1e6
    vom = base.vom_annual_usd.values / 1e6
    fuel = base.fuel_annual_usd.values / 1e6
    grid = base.grid_annual_usd.values / 1e6  # may be negative (export revenue)
    tac = base.tac_usd_per_yr.values / 1e6
    premium = base.premium_pct.values

    # Positive grid (imports) stacks above; negative grid (exports) shows below 0
    grid_pos = np.where(grid > 0, grid, 0.0)
    grid_neg = np.where(grid < 0, grid, 0.0)  # already negative

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.85))
    x = np.arange(len(cases))
    bw = 0.62

    # Stack positive components
    bot = np.zeros_like(capex)
    components = [
        ("Capital", capex, MAIN_FILLS[0], 0.92, None),
        ("Fixed O&M", fom, MAIN_FILLS[1], 0.92, None),
        ("Var. O&M", vom, TAC_STACK_VOM_FILL, 0.92, None),
        ("Fuel", fuel, TAC_STACK_FUEL_FILL, 0.88, None),
    ]
    for lab, vals, color, alpha, hatch in components:
        if vals.sum() < 1e-3:
            continue
        ax.bar(x, vals, bottom=bot, width=bw, color=color, edgecolor=BLOCK_EDGE,
               linewidth=EDGE_LW, alpha=alpha, hatch=hatch, label=lab)
        bot = bot + vals

    # C0 is the only positive grid-import case; render it as a pure lilac fill
    # for this palette trial.
    c0_import = np.where(cases == 0, grid_pos, 0.0)
    other_import = np.where(cases != 0, grid_pos, 0.0)
    if other_import.sum() >= 1e-3:
        ax.bar(x, other_import, bottom=bot, width=bw, color=MAIN_FILLS[0],
               edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, alpha=0.52,
               hatch="////", label="Grid import")
    if c0_import.sum() >= 1e-3:
        ax.bar(x, c0_import, bottom=bot, width=bw,
               color=TAC_STACK_C0_IMPORT_FILL, edgecolor=BLOCK_EDGE,
               linewidth=EDGE_LW, alpha=1.0, label="Grid import")
    bot = bot + grid_pos

    # Below-zero grid export bar (revenue offset)
    if (grid_neg < 0).any():
        ax.bar(x, grid_neg, width=bw,
               color=GRID_EXPORT_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
               alpha=GRID_EXPORT_ALPHA,
               label="Grid export")

    # Section 45U production tax credit (negative; stacks below zero with export).
    ptc = np.nan_to_num(base["ptc_annual_usd"].to_numpy(dtype=float)) / 1e6
    if (ptc < 0).any():
        ax.bar(x, ptc, bottom=grid_neg, width=bw,
               color="#7FB7A6", edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
               alpha=0.85, label="PTC")

    # Net TAC line marker on top of each bar
    stack_top = capex + fom + vom + fuel + grid_pos
    for xi, tac_i, prem_i, top in zip(x, tac, premium, stack_top):
        ax.scatter(xi, tac_i, marker="_", s=320, color=MAIN_STROKES[3], linewidth=1.6,
                   zorder=5, label="Net TAC" if xi == 0 else None)
        # Place the label ABOVE the visible stack top (not the net TAC) so it
        # never overlaps the bars themselves.
        ax.annotate(
            f"TAC ${tac_i:.0f} M\nP = {prem_i:+.0f}%",
            xy=(xi, top),
            xytext=(0, VALUE_LABEL_OFFSET_PT),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=8,
            color="#000000",
        )

    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylabel(r"Annualised cost (M\$/yr)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    legend_handles = [
        Patch(facecolor=MAIN_FILLS[0], edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              label="CapEx"),
        Patch(facecolor=MAIN_FILLS[1], edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              label="FOM"),
        Patch(facecolor=TAC_STACK_VOM_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              label="VOM"),
        Patch(facecolor=TAC_STACK_FUEL_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              alpha=0.88, label="Fuel"),
        Patch(facecolor=TAC_STACK_C0_IMPORT_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              label="Import"),
        Patch(facecolor=GRID_EXPORT_FILL, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              alpha=GRID_EXPORT_ALPHA, label="Export"),
        Patch(facecolor="#7FB7A6", edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
              alpha=0.85, label="PTC"),
        Line2D([0], [0], color=MAIN_STROKES[3], lw=1.6, label="Net TAC"),
    ]
    ax.legend(handles=legend_handles, loc="upper left",
              bbox_to_anchor=(0.04, -0.24, 0.92, 0.08), mode="expand",
              ncol=8, frameon=False, fontsize=8, columnspacing=0.15,
              handlelength=0.8, handletextpad=0.25, borderaxespad=0.0)

    fig.tight_layout()
    save_triplet(fig, "fig2_tac_stack", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_3_tac_stack()

In [ ]:
# Figure 4: Data-center size matching sensitivity (fig12_s8_size_matching)
# Panels A-D: premium, TAC gap, IT-cost intensity, and net grid export.
def figure_4_size_matching() -> None:
    s8 = df[df.group == "s8_size_matching"].copy()
    if s8.empty:
        print("[fig12] missing s8_size_matching rows; skipping")
        return

    s8["avg_it_mw"] = s8.it_energy_annual_MWh / 8760.0
    s8["net_export_twh"] = (
        s8.P_grid_sell_annual_MWh.fillna(0.0)
        - s8.P_grid_buy_annual_MWh.fillna(0.0)
    ) / 1e6
    s8["tac_myr"] = s8.tac_usd_per_yr / 1e6
    s8["grid_myr"] = s8.tac_case0_baseline_usd_per_yr / 1e6
    s8["gap_myr"] = s8.tac_myr - s8.grid_myr
    s8["tac_per_mwh_it"] = s8.tac_usd_per_yr / s8.it_energy_annual_MWh
    s8["grid_per_mwh_it"] = s8.tac_case0_baseline_usd_per_yr / s8.it_energy_annual_MWh
    s8["gap_per_mwh_it"] = s8.tac_per_mwh_it - s8.grid_per_mwh_it

    fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 4.85), sharex=True)

    line_palette = {
        0: PALETTE["fill_blue"],
        1: PALETTE["fill_blue"],
        2: PALETTE["fill_salmon"],
        3: PALETTE["fill_orange"],
    }
    marker_face = "#7A7A7A"
    marker_edge = "#FFFFFF"
    line_lw = LINE_PLOT_LW
    marker_size = 5.6
    marker_edge_lw = 1.05
    fig4_marker = {1: "X", 2: "D", 3: "^"}
    x_tick_data = (
        s8[s8.case_id == 2]
        .sort_values("load_multiplier")[["load_multiplier", "avg_it_mw"]]
        .drop_duplicates()
    )
    x_ticks = x_tick_data.avg_it_mw.to_numpy()
    x_tick_labels = [f"{x:.0f}" for x in x_ticks]

    def plot_line(ax, x, y, cid, label, marker=None, **kwargs):
        ax.plot(
            x,
            y,
            color=line_palette[cid],
            linewidth=line_lw,
            marker=marker or CASE_MARKER.get(cid, "o"),
            markersize=marker_size,
            markerfacecolor=marker_face,
            markeredgecolor=marker_edge,
            markeredgewidth=marker_edge_lw,
            label=label,
            **kwargs,
        )

    # (a) Cost premium as the campus grows against a fixed BWRX-300.
    ax = axes[0, 0]
    for cid in (1, 2, 3):
        sub = s8[s8.case_id == cid].sort_values("load_multiplier")
        style = {}
        if cid == 1:
            style.update(linestyle="--", zorder=5)
        elif cid == 2:
            style.update(zorder=4)
        else:
            style.update(zorder=3)
        plot_line(
            ax,
            sub.avg_it_mw,
            sub.premium_pct,
            cid,
            CASE_LABEL[cid].replace(" — ", "\n"),
            marker=fig4_marker[cid],
            **style,
        )
    ax.axhline(0, **ZERO_LINE_KW)
    ax.axvline(94.7, color=PALETTE["accent_sky"], linewidth=0.9,
               linestyle="--", alpha=0.95)
    ax.text(94.7, -438, "1x load", ha="center", va="bottom",
            fontsize=8, color="#333333", rotation=90)
    ax.set_ylabel("Grid-cost margin (%)")
    ax.set_xlim(35, 300)
    ax.set_ylim(-465, 45)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="lower right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure 4, Panel A: grid-cost margin versus campus scale.
    add_panel_label(ax, "a")

    # (b) Absolute TAC rises with size, but so does the matching grid baseline.
    ax = axes[0, 1]
    c2 = s8[s8.case_id == 2].sort_values("load_multiplier")
    plot_line(ax, c2.avg_it_mw, c2.grid_myr, 1, "Grid-only baseline", marker="o")
    plot_line(ax, c2.avg_it_mw, c2.tac_myr, 2, "Case 2 TAC", marker="D")
    fill_between_vertical_gradient(ax, c2.avg_it_mw, c2.grid_myr, c2.tac_myr,
                                   PALETTE["fill_salmon"], alpha=0.32)
    gap_25_myr = c2.loc[c2.load_multiplier == 2.5, "gap_myr"].iloc[0]
    ax.annotate(
        f"{gap_25_myr:.0f} M\\$/yr\ngap at 2.5x",
        xy=(c2.loc[c2.load_multiplier == 2.5, "avg_it_mw"].iloc[0],
            c2.loc[c2.load_multiplier == 2.5, "tac_myr"].iloc[0]),
        xytext=(-22, -34),
        textcoords="offset points",
        fontsize=TEXT_SIZE,
        color="#333333",
        ha="right",
        va="top",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7),
    )
    ax.set_ylabel("Annual TAC (M$/yr)")
    ax.set_ylim(0, max(c2.tac_myr.max(), c2.grid_myr.max()) * 1.08)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper left", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure 4, Panel B: Case 2 TAC versus matching grid-only TAC.
    add_panel_label(ax, "b")

    # (c) Unit-load cost intensity reveals the utilization-efficiency gain.
    ax = axes[1, 0]
    plot_line(ax, c2.avg_it_mw, c2.grid_per_mwh_it, 1, "Grid-only baseline", marker="o")
    plot_line(ax, c2.avg_it_mw, c2.tac_per_mwh_it, 2, "Case 2 TAC", marker="D")
    fill_between_vertical_gradient(ax, c2.avg_it_mw, c2.grid_per_mwh_it, c2.tac_per_mwh_it,
                                   PALETTE["fill_salmon"], alpha=0.32)
    gap_1_intensity = c2.loc[c2.load_multiplier == 1.0, "gap_per_mwh_it"].iloc[0]
    gap_25_intensity = c2.loc[c2.load_multiplier == 2.5, "gap_per_mwh_it"].iloc[0]
    ax.annotate(
        f"{gap_1_intensity:.0f} -> {gap_25_intensity:.0f}\nUSD per MWh IT gap",
        xy=(c2.loc[c2.load_multiplier == 2.5, "avg_it_mw"].iloc[0],
            c2.loc[c2.load_multiplier == 2.5, "tac_per_mwh_it"].iloc[0]),
        xytext=(-12, 36),
        textcoords="offset points",
        fontsize=TEXT_SIZE,
        color="#333333",
        ha="right",
        va="center",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7),
    )
    ax.set_xlabel(r"Mean IT load (MW$_\mathrm{e}$)")
    ax.set_ylabel(r"Cost intensity (\$/MWh$_\mathrm{IT}$)")
    ax.set_ylim(55, 440)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure 4, Panel C: cost intensity per MWh of IT load.
    add_panel_label(ax, "c")

    # (d) Merchant export falls as the campus absorbs more of the fixed unit.
    ax = axes[1, 1]
    for cid in (1, 2):
        sub = s8[s8.case_id == cid].sort_values("load_multiplier")
        plot_line(ax, sub.avg_it_mw, sub.net_export_twh, cid, f"C{cid}", marker=fig4_marker[cid])
    ax.axhline(0, **ZERO_LINE_KW)
    balance = s8[(s8.case_id == 2) & (s8.load_multiplier == 2.5)].iloc[0]
    ax.axvline(balance.avg_it_mw, color=PALETTE["accent_sky"], linewidth=0.9,
               linestyle="--", alpha=0.95)
    ax.annotate(
        "2.5x: near\nexport balance",
        xy=(balance.avg_it_mw, balance.net_export_twh),
        xytext=(248, 0.86),
        textcoords="data",
        fontsize=TEXT_SIZE,
        color="#333333",
        ha="left",
        va="center",
        arrowprops=dict(arrowstyle="->", color="#555555", lw=0.7),
    )
    ax.set_xlabel(r"Mean IT load (MW$_\mathrm{e}$)")
    ax.set_ylabel("Net grid export (TWh/yr)")
    ax.set_ylim(-1.10, 2.08)
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="upper right", frameon=True, facecolor="white",
              edgecolor="none", framealpha=0.88, fontsize=8,
              borderpad=0.22, handlelength=1.05)
    # Figure 4, Panel D: net grid export versus campus scale.
    add_panel_label(ax, "d")

    for ax in axes.flat:
        ax.set_xlim(35, 300)
        ax.set_xticks(x_ticks)
        ax.tick_params(direction="out", length=3)
    for ax in axes[1, :]:
        ax.set_xticklabels(x_tick_labels)

    fig.tight_layout(w_pad=1.35, h_pad=1.05)
    save_triplet(fig, "fig12_s8_size_matching", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_4_size_matching()


In [ ]:
# Figure 5: Cost-carbon tradeoff and nuclear CO2 accounting (fig5_cost_carbon_accounting)
# Panels A-B: cost-carbon plane; nuclear carbon-accounting components.
def figure_5_cost_carbon_accounting() -> None:
    s2 = df[df.group == "s2_price"].copy().sort_values(["case_id", "year"])
    s2["tac_m"] = s2.tac_usd_per_yr / 1e6
    s2["co2_kt"] = s2.co2_annual_tonnes / 1e3

    years = [2022, 2023, 2024]
    cases = [0, 1, 2, 3]
    local_case_label = {
        0: "C0 Grid",
        1: "C1 Nuclear",
        2: "C2 Nuclear + extraction",
        3: "C3 NGCC",
    }
    local_case_short = {0: "C0", 1: "C1", 2: "C2", 3: "C3"}
    local_case_marker = {0: "o", 1: "s", 2: "P", 3: "^"}
    year_color = {
        2022: PALETTE["fill_blue"],
        2023: PALETTE["fill_salmon"],
        2024: PALETTE["fill_orange"],
    }

    def finish_local_axes(ax, grid_axis="y"):
        ax.set_axisbelow(True)
        ax.grid(axis=grid_axis, alpha=0.30, linestyle="--", linewidth=0.5)
        ax.tick_params(direction="out", length=3)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig, axes = plt.subplots(
        1, 2, figsize=(BODY_W, 3.62), constrained_layout=True,
        gridspec_kw={"width_ratios": [1.0, 1.08], "wspace": 0.035},
    )

    ax = axes[0]
    ax.set_box_aspect(1)
    for cid in cases:
        sub = s2[s2.case_id == cid].sort_values("year")
        ax.plot(sub.tac_m, sub.co2_kt, color="#B6B6B6",
                linewidth=THIN_LINE_PLOT_LW, alpha=0.68, zorder=1.5)
        for _, row in sub.iterrows():
            ax.scatter(row.tac_m, row.co2_kt,
                       marker=local_case_marker[cid], s=LINE_MARKER_AREA,
                       facecolor=year_color[int(row.year)],
                       edgecolor=BLOCK_EDGE, linewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)

    ax.set_xlabel(r"TAC (M\$/yr)")
    ax.set_ylabel(r"Annual CO$_2$ (kt CO$_2$/yr)")
    ax.set_xlim(20, 260)
    ax.set_ylim(-525, 520)
    finish_local_axes(ax)

    case_handles = [
        Line2D([0], [0], marker=local_case_marker[cid], linestyle="none",
               color="none", markerfacecolor="#777777",
               markeredgecolor=BLOCK_EDGE, markeredgewidth=LINE_MARKER_EDGE_WIDTH,
               markersize=LINE_MARKER_SIZE,
               label=local_case_label[cid])
        for cid in cases
    ]
    year_handles = [
        Line2D([0], [0], marker="o", linestyle="none", color="none",
               markerfacecolor=year_color[year], markeredgecolor=BLOCK_EDGE,
               markeredgewidth=LINE_MARKER_EDGE_WIDTH,
               markersize=LINE_MARKER_SIZE, label=str(year))
        for year in years
    ]
    leg_case = ax.legend(
        handles=case_handles, loc="upper right", bbox_to_anchor=(0.985, 0.985),
        ncol=1, frameon=True, facecolor="white", edgecolor="none",
        framealpha=0.90, fontsize=TEXT_SIZE, handlelength=1.05,
        handletextpad=0.60, borderpad=0.42, labelspacing=0.50,
        borderaxespad=0.0,
    )
    ax.add_artist(leg_case)
    ax.legend(
        handles=year_handles, loc="center", bbox_to_anchor=(0.52, 0.55),
        ncol=3, frameon=True, facecolor="white", edgecolor="none",
        framealpha=0.86, fontsize=TEXT_SIZE, handlelength=0.8,
        handletextpad=0.35, borderpad=0.28, columnspacing=0.75,
        title="Year", title_fontsize=TEXT_SIZE,
    )

    ax.annotate("", xy=(0.135, 0.105), xytext=(0.235, 0.155),
                xycoords="axes fraction",
                arrowprops=dict(arrowstyle="->", color=PALETTE["stroke_teal"],
                                lw=0.9, alpha=0.72))
    ax.text(0.245, 0.165, "preferred", transform=ax.transAxes,
            fontsize=TEXT_SIZE, color=PALETTE["stroke_teal"], style="italic",
            ha="left", va="center")
    add_panel_label(ax, "a", x=-0.12, y=1.04)

    ax = axes[1]
    x_pos = []
    x_labels = []
    grid_import = []
    reactor_lca = []
    export_credit = []
    net_co2 = []
    xpos = 0.0
    for year in years:
        for cid in (1, 2):
            row = s2[(s2.year == year) & (s2.case_id == cid)].iloc[0]
            x_pos.append(xpos)
            x_labels.append(f"{str(year)[-2:]}\n{local_case_short[cid]}")
            grid_import.append(row.co2_grid_import_tonnes / 1e3)
            reactor_lca.append(row.co2_rx_lifecycle_tonnes / 1e3)
            export_credit.append(-row.co2_export_credit_tonnes / 1e3)
            net_co2.append(row.co2_kt)
            xpos += 1.0
        xpos += 0.45

    x_pos = np.array(x_pos)
    grid_import = np.array(grid_import)
    reactor_lca = np.array(reactor_lca)
    export_credit = np.array(export_credit)
    net_co2 = np.array(net_co2)
    positive_total = grid_import + reactor_lca

    bars_grid = ax.bar(x_pos, grid_import, color=PALETTE["fill_blue"],
                       edgecolor=BLOCK_EDGE, linewidth=BLOCK_EDGE_LW, width=0.70,
                       label="Grid import")
    apply_vertical_bar_gradients(ax, bars_grid, PALETTE["fill_blue"])
    bars_lca = ax.bar(x_pos, reactor_lca, bottom=grid_import,
                      color=PALETTE["fill_orange"], edgecolor=BLOCK_EDGE,
                      linewidth=BLOCK_EDGE_LW, width=0.70, label="Reactor LCA")
    apply_vertical_bar_gradients(ax, bars_lca, PALETTE["fill_orange"])
    bars_export = ax.bar(x_pos, export_credit, color=PALETTE["fill_salmon"],
                         edgecolor=BLOCK_EDGE, linewidth=BLOCK_EDGE_LW, width=0.70,
                         label="Export credit")
    apply_vertical_bar_gradients(ax, bars_export, PALETTE["fill_salmon"], white_at_zero=True, gamma=3.0)
    ax.scatter(x_pos, net_co2, marker="_", s=170, color="#222222",
               linewidth=1.0, zorder=5, label="Net")

    for xi, top, net in zip(x_pos, positive_total, net_co2):
        annotate_vertical_value(ax, xi, top, f"+{top:.0f}", color="#111111")
        annotate_vertical_value(ax, xi, net, f"{net:.0f}", color="#111111", offset_pt=POINT_LABEL_OFFSET_PT)

    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(x_labels)
    ax.set_ylabel(r"Nuclear CO$_2$ accounting (kt/yr)")
    ax.set_ylim(-835, 500)
    finish_local_axes(ax)
    legend_handles = [
        Patch(facecolor=PALETTE["fill_blue"], edgecolor=BLOCK_EDGE,
              label="Grid import"),
        Patch(facecolor=PALETTE["fill_orange"], edgecolor=BLOCK_EDGE,
              label="Reactor LCA"),
        Patch(facecolor=PALETTE["fill_salmon"], edgecolor=BLOCK_EDGE,
              label="Export credit"),
        Line2D([0], [0], color="#222222", linewidth=1.2, label="Net"),
    ]
    ax.legend(handles=legend_handles, loc="upper center",
              bbox_to_anchor=(0.50, 1.095), ncol=4, frameon=False,
              fontsize=TEXT_SIZE, handlelength=0.78, columnspacing=0.52,
              handletextpad=0.34, borderaxespad=0.0)
    add_panel_label(ax, "b", x=-0.12, y=1.04)

    save_triplet(fig, "fig5_cost_carbon_accounting", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_5_cost_carbon_accounting()

In [ ]:
# Figure 6: Hourly dispatch of Case 2 in winter and summer weeks (fig4_case2_dispatch)
# Panels A-B: winter week and summer week dispatch.
def figure_6_case2_dispatch() -> None:
    disp = pd.read_csv(OUTPUTS / "main_baseline" / "case2" / "dispatch.csv.gz")
    winter_start = 24 * 14            # Jan 15 00:00 (hour 336)
    summer_start = 24 * (31 + 28 + 31 + 30 + 31 + 30 + 14)  # July 15 00:00 = hour 4680
    weeks = [
        ("Winter (Jan 15 – Jan 22)", winter_start),
        ("Summer (Jul 15 – Jul 22)", summer_start),
    ]

    fig, axes = plt.subplots(2, 1, figsize=(BODY_W, 4.4), sharex=False)

    handle_specs: list[tuple[str, str, str, str]] = []  # (label, kind, color, ls)
    for ax, (name, start) in zip(axes, weeks):
        sl = disp.iloc[start:start + 168].reset_index(drop=True)
        hours = np.arange(168)

        # Primary axis (left): reactor thermal + turbine net electric power.
        ax.fill_between(hours, 0, sl.P_turb_net_MW,
                        color=PALETTE["fill_blue"], alpha=0.85,
                        edgecolor="white", linewidth=0.4,
                        label=r"$P_\mathrm{turb,net}$ (MW$_\mathrm{e}$)")
        ax.plot(hours, sl.P_rx_MWth, color=FIGURE6_LINE_COLORS[0], linewidth=THIN_LINE_PLOT_LW,
                label=r"$P_\mathrm{rx}$ (MW$_\mathrm{th}$)")
        ax.set_xlim(0, 167)
        ax.set_xlabel("Hour of week")
        ax.set_ylabel(r"Power / heat (MW)")
        ax.set_axisbelow(True)
        ax.grid(**GRID_KW)

        # Secondary axis (right): cooling streams + extraction flow.
        ax_r = ax.twinx()
        ax_r.plot(hours, sl.Q_to_abs_MWth,
                  color=FIGURE6_LINE_COLORS[1], linewidth=THIN_LINE_PLOT_LW, linestyle="--",
                  label=r"$Q_\mathrm{extract}$ (MW$_\mathrm{th}$)")
        ax_r.plot(hours, sl.Q_abs_cool_MWth,
                  color=FIGURE6_LINE_COLORS[2], linewidth=THIN_LINE_PLOT_LW,
                  label=r"$Q_\mathrm{abs,cool}$ (MW$_\mathrm{c}$)")
        ax_r.fill_between(hours, 0, sl.Q_VCC_cool_MWth,
                          color=PALETTE["fill_salmon"], alpha=0.6,
                          edgecolor="white", linewidth=0.3,
                          label=r"$Q_\mathrm{VCC,backup}$ (MW$_\mathrm{c}$)")
        ax_r.set_ylabel(r"Cooling / extraction (MW)")
        ax_r.set_ylim(bottom=0)
        ax_r.spines["right"].set_visible(True)

        ax.text(0.02, 0.08, name, transform=ax.transAxes, fontsize=8,
                fontweight="bold", color="#222222",
                bbox=dict(facecolor="white", edgecolor="none", alpha=0.78, pad=1.2))

        # Collect handles once (winter has all)
        if not handle_specs:
            handle_specs = [
                (r"$P_\mathrm{turb,net}$ (MW$_\mathrm{e}$)", "patch",
                 PALETTE["fill_blue"], None),
                (r"$P_\mathrm{rx}$ (MW$_\mathrm{th}$)", "line",
                 FIGURE6_LINE_COLORS[0], "-"),
                (r"$Q_\mathrm{extract}$ (MW$_\mathrm{th}$)", "line",
                 FIGURE6_LINE_COLORS[1], "--"),
                (r"$Q_\mathrm{abs,cool}$ (MW$_\mathrm{c}$)", "line",
                 FIGURE6_LINE_COLORS[2], "-"),
                (r"$Q_\mathrm{VCC,backup}$ (MW$_\mathrm{c}$)", "patch",
                 PALETTE["fill_salmon"], None),
            ]

    # Single combined legend drawn from synthetic handles
    legend_handles = []
    for label, kind, color, ls in handle_specs:
        if kind == "patch":
            from matplotlib.patches import Patch
            legend_handles.append(Patch(facecolor=color, edgecolor="white",
                                        label=label))
        else:
            legend_handles.append(Line2D([0], [0], color=color, linestyle=ls or "-",
                                         linewidth=THIN_LINE_PLOT_LW, label=label))
    fig.legend(handles=legend_handles, loc="lower center",
               bbox_to_anchor=(0.5, 0.02), ncol=5, frameon=True,
               facecolor="white", edgecolor="none", framealpha=0.86, fontsize=8,
               columnspacing=1.2, handlelength=1.5, borderpad=0.25)
    # Figure 6, Panel A: winter-week Case 2 dispatch.
    add_panel_label(axes[0], "a", x=-0.06, y=1.02)
    # Figure 6, Panel B: summer-week Case 2 dispatch.
    add_panel_label(axes[1], "b", x=-0.06, y=1.02)
    fig.tight_layout()
    fig.subplots_adjust(bottom=0.18)
    save_triplet(fig, "fig4_case2_dispatch", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_6_case2_dispatch()

In [ ]:
# Figure 7: Marginal value decomposition for adding absorption cooling (fig4bis_value_decomp)
# Single-panel manuscript figure.
def figure_7_value_decomp() -> None:
    decomp_path = OUTPUTS / "figures" / "value_decomp_case2.csv"
    if not decomp_path.exists():
        print(f"[fig4bis] missing {decomp_path}; skipping")
        return
    vd = pd.read_csv(decomp_path)
    base_row = vd[vd.matching_key == "main_baseline/base"].iloc[0]

    # Waterfall order: net = sum of these (positive adds, negative subtracts)
    components = [
        ("VCC\nsaved",
         base_row.vcc_elec_saved_usd_per_yr / 1e6),
        ("Power\nlost",
         base_row.turbine_gen_lost_usd_per_yr / 1e6),
        ("Abs.\nCAPEX",
         base_row.absorption_capex_fom_usd_per_yr / 1e6),
        ("VCC\nbackup",
         base_row.crystal_cutoff_backup_usd_per_yr / 1e6),
        ("PTC\nforgone",
         base_row.ptc_forgone_usd_per_yr / 1e6),
        ("Water\ncost",
         base_row.extra_water_cost_usd_per_yr / 1e6),
    ]
    net = base_row.net_value_of_absorption_usd_per_yr / 1e6
    residual = base_row.residual_usd_per_yr / 1e6
    components.append(("Residual", residual))

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.75))
    bar_width = 0.58

    # Cumulative running total for the waterfall
    cum = 0.0
    x_pos = []
    heights = []
    bottoms = []
    colors = []
    for name, val in components:
        x_pos.append(name)
        heights.append(val)
        bottoms.append(cum)
        if name == "Residual":
            colors.append(MAIN_FILLS[3])
        else:
            colors.append(
                MAIN_FILLS[0] if val > 0 else MAIN_FILLS[1]
            )
        cum += val

    ax.bar(
        x_pos, heights, bottom=bottoms, color=colors,
        edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=bar_width, alpha=0.88,
    )

    # Net result as a final summary bar
    x_pos.append("Net vs\nCase 1")
    heights.append(net)
    bottoms.append(0.0)
    net_color = (
        MAIN_FILLS[0] if net > 0 else MAIN_FILLS[1]
    )
    ax.bar(
        ["Net vs\nCase 1"], [net], color=net_color,
        edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=bar_width, alpha=0.88,
    )

    # Light waterfall connectors show the running total between adjacent bars.
    for i in range(len(x_pos) - 1):
        level = bottoms[i] + heights[i]
        ax.plot(
            [i + bar_width / 2, i + 1 - bar_width / 2], [level, level],
            color="#BDBDBD", linewidth=0.55, alpha=0.85,
            solid_capstyle="butt", zorder=2.2,
        )

    # Annotate all components. Major terms use one decimal; the two small
    # accounting terms use leader lines and two decimals so they are not
    # mistaken for missing data in the printed figure.
    small_label_offsets = {
        "VCC\nbackup": (-3, 3),
        "PTC\nforgone": (-3, 10),
        "Water\ncost": (3, 8),
    }
    value_label_fontsize = 8.0
    for i, (x, h, b) in enumerate(zip(x_pos[:-1], heights[:-1], bottoms[:-1])):
        y = b + h
        if abs(h) >= 0.5:
            if x == "Residual":
                ax.text(
                    i, b + h / 2, f"\\${h:+.1f}M",
                    ha="center", va="center", fontsize=value_label_fontsize, color="#000000",
                )
            else:
                force_positive_side = h >= 0
                if x == "Abs.\nCAPEX" and y > 0:
                    force_positive_side = True
                annotate_vertical_value(
                    ax, i, y, f"\\${h:+.1f}M",
                    fontsize=value_label_fontsize,
                    force_positive_side=force_positive_side,
                )
        else:
            dx, dy = small_label_offsets.get(x, (10, 10))
            ax.annotate(
                f"\\${h:+.2f}M",
                xy=(i, y), xycoords="data",
                xytext=(dx, dy), textcoords="offset points",
                ha="center", va="bottom", fontsize=value_label_fontsize, color="#000000",
                zorder=10,
                arrowprops=dict(
                    arrowstyle="-", color="#666666", linewidth=0.55,
                    shrinkA=0, shrinkB=0,
                ),
            )

    # Final summary bar label
    net_color_label = "#000000"
    annotate_vertical_value(
        ax, len(x_pos) - 1, net, f"\\${net:+.1f}M",
        fontsize=value_label_fontsize, color=net_color_label,
    )

    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_ylabel("Δ vs Case 1 TAC (M\\$/yr)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    plt.setp(ax.get_xticklabels(), fontsize=8)
    # Give some headroom so the leader-line text has somewhere to live
    ymin = min(min(bottoms), -6.0)
    ymax = max(heights) + 1.2
    ax.set_ylim(ymin, ymax)

    fig.tight_layout()
    save_triplet(fig, "fig4bis_value_decomp", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_7_value_decomp()

In [ ]:
# Figure 8: Cooling-side response of the absorption configuration (fig_sensitivity_cooling_response)
# Panels A-D: C2-C1 TAC gap, cooling-electricity saving, PUE premium gap, and carbon-price premium gap.
def figure_8_cooling_response() -> None:
    pue_levels = [1.10, 1.30, 1.50]
    s1 = df[df.group == "s1_pue"].copy()
    s6 = df[df.group == "s6_carbon_price"].copy()

    tac_gap = []
    pue_gap = []
    cooling_elec_saving = []
    vd = pd.read_csv(FIGURES / "value_decomp_case2.csv")
    for pue in pue_levels:
        c1 = s1[(s1.case_id == 1) & (s1.pue.round(2) == pue)].iloc[0]
        c2 = s1[(s1.case_id == 2) & (s1.pue.round(2) == pue)].iloc[0]
        vrow = vd[vd.matching_key == f"s1_pue/_pue{int(round(pue * 100)):03d}"].iloc[0]
        tac_gap.append((c2.tac_usd_per_yr - c1.tac_usd_per_yr) / 1e6)
        pue_gap.append((c2.heat_recovery_premium - c1.heat_recovery_premium) * 100.0)
        # Explicit net column prevents accidentally subtracting Case 2 backup/top-up twice.
        cooling_elec_saving.append(vrow.net_vcc_elec_saved_usd_per_yr / 1e6)

    co2_levels = [0.0, 50.0, 100.0]
    co2_gap = []
    for price in co2_levels:
        c1 = s6[(s6.case_id == 1) & (s6.carbon_price_usd_per_tco2 == price)].iloc[0]
        c2 = s6[(s6.case_id == 2) & (s6.carbon_price_usd_per_tco2 == price)].iloc[0]
        co2_gap.append((c2.heat_recovery_premium - c1.heat_recovery_premium) * 100.0)

    fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 4.65), constrained_layout=True)
    ax_a, ax_b, ax_c, ax_d = axes.ravel()
    x = np.arange(len(pue_levels))
    xtick = [f"{p:.2f}" for p in pue_levels]

    tac_colors = [MAIN_FILLS[1] if v >= 0 else MAIN_FILLS[0] for v in tac_gap]
    bars = ax_a.bar(x, tac_gap, color=tac_colors, edgecolor=BLOCK_EDGE,
                    linewidth=EDGE_LW, width=0.58)
    apply_vertical_bar_gradients(ax_a, bars, tac_colors, white_at_zero=True)
    ax_a.axhline(0, **ZERO_LINE_KW)
    for b, v in zip(bars, tac_gap):
        annotate_vertical_value(ax_a, b.get_x() + b.get_width() / 2, v, f"{v:.1f}", fontsize=8)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels(xtick)
    ax_a.set_xlim(-0.5, len(pue_levels) - 0.5)
    ax_a.set_xlabel("PUE")
    ax_a.set_ylabel("C2 - C1 TAC (M\\$/yr)")
    ax_a.set_ylim(min(tac_gap) - 2.8, max(tac_gap) + 2.2)
    # Figure 8, Panel A: Case 2 minus Case 1 TAC penalty across PUE.
    add_panel_label(ax_a, "a", x=-0.13, y=1.04)

    bars = ax_b.bar(x, cooling_elec_saving, color=MAIN_FILLS[0], edgecolor=BLOCK_EDGE,
                    linewidth=EDGE_LW, width=0.58)
    apply_vertical_bar_gradients(ax_b, bars, MAIN_FILLS[0])
    for b, v in zip(bars, cooling_elec_saving):
        annotate_vertical_value(ax_b, b.get_x() + b.get_width() / 2, v, f"{v:.1f}", fontsize=8)
    ax_b.set_xticks(x)
    ax_b.set_xticklabels(xtick)
    ax_b.set_xlim(-0.5, len(pue_levels) - 0.5)
    ax_b.set_xlabel("PUE")
    ax_b.set_ylabel("Net cooling-electricity\nsaving (M\\$/yr)")
    ax_b.set_ylim(0, max(cooling_elec_saving) * 1.28)
    # Figure 8, Panel B: VCC electricity saving net of backup top-up in Case 2.
    add_panel_label(ax_b, "b", x=-0.13, y=1.04)

    line_blue = PALETTE["accent_sky"]
    line_pink = PALETTE["fill_salmon"]
    marker_gray = PALETTE["fill_gray"]
    ax_c.plot(x, pue_gap, "-", color=line_blue, linewidth=LINE_PLOT_LW,
              marker="o", markersize=LINE_MARKER_SIZE, markerfacecolor=marker_gray,
              markeredgecolor=BLOCK_EDGE, markeredgewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)
    ax_c.axhline(0, **ZERO_LINE_KW)
    for xi, yi in zip(x, pue_gap):
        annotate_vertical_value(ax_c, xi, yi, f"{yi:+.1f}", fontsize=8, offset_pt=POINT_LABEL_OFFSET_PT)
    ax_c.set_xticks(x)
    ax_c.set_xticklabels(xtick)
    ax_c.set_xlabel("PUE")
    ax_c.set_ylabel("C2 - C1 margin gap (pp)")
    ax_c.set_xlim(-0.35, len(pue_levels) - 0.65)
    # Figure 8, Panel C: Case 2 minus Case 1 premium gap across PUE.
    add_panel_label(ax_c, "c", x=-0.13, y=1.04)

    x2 = np.arange(len(co2_levels))
    ax_d.plot(x2, co2_gap, "-", color=line_pink, linewidth=LINE_PLOT_LW,
              marker="o", markersize=LINE_MARKER_SIZE, markerfacecolor=marker_gray,
              markeredgecolor=BLOCK_EDGE, markeredgewidth=LINE_MARKER_EDGE_WIDTH, zorder=3)
    ax_d.axhline(0, **ZERO_LINE_KW)
    for xi, yi in zip(x2, co2_gap):
        annotate_vertical_value(ax_d, xi, yi, f"{yi:+.1f}", fontsize=8, offset_pt=POINT_LABEL_OFFSET_PT)
    ax_d.set_xticks(x2)
    ax_d.set_xticklabels([f"{int(c)}" for c in co2_levels])
    ax_d.set_xlabel(r"Carbon price (\$/tCO$_2$)")
    ax_d.set_ylabel("C2 - C1 margin gap (pp)")
    ax_d.set_xlim(-0.35, len(co2_levels) - 0.65)
    # Figure 8, Panel D: Case 2 minus Case 1 premium gap across carbon price.
    add_panel_label(ax_d, "d", x=-0.13, y=1.04)

    ymin = min(min(pue_gap), min(co2_gap)) - 3.8
    ymax = max(max(pue_gap), max(co2_gap)) + 2.4
    for ax in (ax_c, ax_d):
        ax.set_ylim(ymin, ymax)

    for ax in axes.ravel():
        ax.set_axisbelow(True)
        ax.grid(**GRID_KW)
        ax.tick_params(direction="out", length=3)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    save_triplet(fig, "fig_sensitivity_cooling_response", str(FIGURES))
    display(fig)
    plt.close(fig)


figure_8_cooling_response()

In [ ]:
# Figure 9: Market, reactor-capital, financing, and joint capital-cost boundaries (fig_sensitivity_boundary_atlas)
# Panels A-D: ERCOT year regime, SMR CAPEX trajectory, WACC, and joint SMR x absorption CAPEX.
def figure_9_boundary_atlas() -> None:
    fig = plt.figure(figsize=(BODY_W, 6.35))
    gs = fig.add_gridspec(
        3, 2,
        height_ratios=[1.0, 0.88, 1.0],
        left=0.09, right=0.98, top=0.98, bottom=0.09,
        hspace=0.44, wspace=0.30,
    )
    ax_a = fig.add_subplot(gs[0, :])
    ax_b = fig.add_subplot(gs[1, 0])
    ax_c = fig.add_subplot(gs[1, 1])
    ax_d_frame = fig.add_subplot(gs[2, :])
    ax_d_frame.set_axis_off()
    ax_d_frame.patch.set_alpha(0.0)
    ax_d_frame.set_zorder(5)
    d_box = gs[2, :].get_position(fig)
    cbar_gap = 0.012
    cbar_w = 0.018
    cbar_label_space = 0.070
    ax_d = fig.add_axes([
        d_box.x0,
        d_box.y0,
        d_box.width - cbar_gap - cbar_w - cbar_label_space,
        d_box.height,
    ])
    cax_d = fig.add_axes([
        d_box.x1 - cbar_label_space - cbar_w,
        d_box.y0,
        cbar_w,
        d_box.height,
    ])

    case_fill = {
        0: MAIN_FILLS[3],
        1: MAIN_FILLS[0],
        2: MAIN_FILLS[1],
        3: MAIN_FILLS[2],
    }

    def label_bars(ax, bars, vals, fontsize=TEXT_SIZE, outside_below_labels=None, outside_above_labels=None):
        for b, v in zip(bars, vals):
            annotate_vertical_value(
                ax,
                b.get_x() + b.get_width() / 2,
                v,
                f"{v:.0f}",
                fontsize=fontsize,
            )

    # a) ERCOT year regime: four cases in each market year.
    s2 = df[df.group == "s2_price"].copy().sort_values(["year", "case_id"])
    years = sorted(s2.year.unique())
    cases = sorted(s2.case_id.unique())
    x = np.arange(len(years))
    bar_w = 0.18
    for i, cid in enumerate(cases):
        vals = [
            s2[(s2.year == y) & (s2.case_id == cid)].premium_pct.iloc[0]
            for y in years
        ]
        offset = (i - (len(cases) - 1) / 2) * bar_w
        bars = ax_a.bar(
            x + offset,
            vals,
            width=bar_w,
            color=case_fill[cid],
            edgecolor=BLOCK_EDGE,
            linewidth=EDGE_LW,
            label=CASE_LABEL[cid].replace(" — ", ": "),
        )
        apply_vertical_bar_gradients(ax_a, bars, case_fill[cid], white_at_zero=True)
        label_bars(ax_a, bars, vals, outside_below_labels={-10, -70}, outside_above_labels={17})
    ax_a.axhline(0, **ZERO_LINE_KW)
    ax_a.set_xticks(x)
    ax_a.set_xticklabels([str(y) for y in years])
    ax_a.set_xlim(x[0] - len(cases) * bar_w / 2 - 0.14,
                  x[-1] + len(cases) * bar_w / 2 + 0.14)
    ax_a.set_xlabel("ERCOT year", labelpad=2)
    ax_a.set_ylabel("Grid-cost margin (%)")
    ax_a.set_ylim(-735, 45)
    ax_a.set_axisbelow(True)
    ax_a.grid(**GRID_KW)
    legend_handles = [
        Patch(facecolor=case_fill[cid], edgecolor=BLOCK_EDGE,
              label=CASE_LABEL[cid].replace(" — ", ": "))
        for cid in cases
    ]
    ax_a.legend(
        handles=legend_handles,
        loc="lower left",
        bbox_to_anchor=(0.01, 0.03),
        ncol=2,
        frameon=False,
        fontsize=8,
        columnspacing=0.8,
        handlelength=1.0,
        borderaxespad=0.0,
    )
    # Figure 9, Panel A: grid-cost margin across ERCOT year regimes.
    add_panel_label(ax_a, "a", x=-0.075, y=1.03)

    # b) SMR CAPEX trajectory: Case 1 and Case 2 move together, but C2 remains lower.
    s4 = df[df.group == "s4_capex"].copy()
    scen_order = ["FOAK", "ATB_Mid", "NOAK"]
    capex_tick = ["FOAK\n\\$14,700", "ATB-Mid\n\\$7,615", "NOAK\n\\$2,250"]
    xb = np.arange(len(scen_order))
    bar_w2 = 0.34
    for i, cid in enumerate((1, 2)):
        vals = [
            s4[(s4.reactor_scenario == s) & (s4.case_id == cid)].premium_pct.iloc[0]
            for s in scen_order
        ]
        offset = (i - 0.5) * bar_w2
        bars = ax_b.bar(
            xb + offset,
            vals,
            width=bar_w2,
            color=MAIN_FILLS[i],
            edgecolor=BLOCK_EDGE,
            linewidth=EDGE_LW,
            label=f"Case {cid}",
        )
        apply_vertical_bar_gradients(ax_b, bars, MAIN_FILLS[i], white_at_zero=True)
        label_bars(ax_b, bars, vals, outside_below_labels={-8, -15})
    ax_b.axhline(0, **ZERO_LINE_KW)
    ax_b.set_xticks(xb)
    ax_b.set_xticklabels(capex_tick)
    ax_b.set_xlim(xb[0] - bar_w2 - 0.14, xb[-1] + bar_w2 + 0.14)
    ax_b.set_xlabel(r"SMR OCC (\$/kW$_\mathrm{e}$)", labelpad=2)
    ax_b.set_ylabel("Grid-cost margin (%)")
    ax_b.set_ylim(-530, 25)
    ax_b.set_axisbelow(True)
    ax_b.grid(**GRID_KW)
    legend_handles = [
        Patch(facecolor=MAIN_FILLS[i], edgecolor=BLOCK_EDGE, label=f"Case {cid}")
        for i, cid in enumerate((1, 2))
    ]
    ax_b.legend(handles=legend_handles, loc="lower right", frameon=False, fontsize=8)
    # Figure 9, Panel B: grid-cost margin across SMR CAPEX trajectories.
    add_panel_label(ax_b, "b", x=-0.16, y=1.04)

    # c) WACC sweep: a focused financing lever for Case 2.
    s7 = df[df.group == "s7_wacc"].copy().sort_values("wacc_effective")
    wacc_labels = [f"{w * 100:.1f}%" for w in s7.wacc_effective.values]
    vals = s7.premium_pct.values
    base_idx = int(np.argmin(np.abs(s7.wacc_effective.values - 0.067)))
    base_val = vals[base_idx]
    colors = [
        MAIN_FILLS[0] if v > base_val else (MAIN_FILLS[1] if v < base_val else MAIN_FILLS[3])
        for v in vals
    ]
    xc = np.arange(len(vals))
    bars = ax_c.bar(
        xc,
        vals,
        width=0.58,
        color=colors,
        edgecolor=BLOCK_EDGE,
        linewidth=EDGE_LW,
    )
    apply_vertical_bar_gradients(ax_c, bars, colors, white_at_zero=True)
    label_bars(ax_c, bars, vals)
    ax_c.axhline(0, **ZERO_LINE_KW)
    ax_c.set_xticks(xc)
    ax_c.set_xticklabels(wacc_labels)
    ax_c.set_xlim(xc[0] - 0.58 / 2 - 0.12, xc[-1] + 0.58 / 2 + 0.12)
    ax_c.set_xlabel("WACC", labelpad=2)
    ax_c.set_ylabel("Grid-cost margin (%)")
    ax_c.set_ylim(-335, 20)
    ax_c.set_axisbelow(True)
    ax_c.grid(**GRID_KW)
    # Figure 9, Panel C: Case 2 grid-cost margin across WACC.
    add_panel_label(ax_c, "c", x=-0.16, y=1.04)

    # d) Joint SMR and absorption-chiller capital-cost frontier.
    s5 = df[df.group == "s5_feasibility_2d"].copy()
    smr_order = ["NOAK", "Low_Mid", "ATB_Mid", "High_Mid", "FOAK"]
    abs_order = ["Bare_Low", "Mid_Low", "Baseline", "Mid_High", "Turnkey_High"]
    smr_vals = [2250, 5000, 7615, 11000, 14700]
    abs_vals = [450, 600, 750, 900, 1200]
    piv = s5.pivot_table(index="smr_capex_tag", columns="absorption_capex_tag", values="premium_pct")
    piv = piv.reindex(smr_order)[abs_order]
    data = piv.values
    from matplotlib.colors import TwoSlopeNorm
    cmap = LinearSegmentedColormap.from_list(
        "hrp_div", [PALETTE["fill_blue"], "#FFFFFF", PALETTE["fill_salmon"]], N=256
    )
    _vmin = float(np.floor(float(data.min()) / 50.0) * 50.0)
    _vmax = max(30.0, float(np.ceil(float(data.max()) / 10.0) * 10.0))
    norm = TwoSlopeNorm(vmin=_vmin, vcenter=0.0, vmax=_vmax)
    im = ax_d.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax_d.text(j, i, f"{data[i, j]:+.0f}%", ha="center", va="center",
                      fontsize=TEXT_SIZE, color="#222222")
    ax_d.set_xticks(np.arange(len(abs_order)))
    ax_d.set_xticklabels([f"{v}" for v in abs_vals])
    ax_d.set_yticks(np.arange(len(smr_order)))
    ax_d.set_yticklabels([f"{t.replace('_', '-')}\n\\${v:,}" for t, v in zip(smr_order, smr_vals)])
    ax_d.set_xlabel(r"Absorption CAPEX (\$/kW$_\mathrm{c}$)", labelpad=2)
    ax_d.set_ylabel(r"SMR CAPEX trajectory (\$/kW$_\mathrm{e}$)")
    ax_d.set_xticks(np.arange(-0.5, len(abs_order), 1), minor=True)
    ax_d.set_yticks(np.arange(-0.5, len(smr_order), 1), minor=True)
    ax_d.grid(which="minor", color=BLOCK_EDGE, linestyle="-", linewidth=1.0)
    ax_d.tick_params(which="minor", bottom=False, left=False)
    cb = fig.colorbar(im, cax=cax_d)
    cb.set_label("Case 2 grid-cost margin (%)", labelpad=5)
    cb.set_ticks([_vmin, -300.0, -200.0, -100.0, 0.0, 10.0, 20.0, _vmax])
    cb.set_ticklabels([f"{int(_vmin):d}", "-300", "-200", "-100", "0", "+10", "+20", f"+{int(_vmax):d}"])
    # Figure 9, Panel D: joint SMR and absorption-chiller capital-cost frontier.
    add_panel_label(ax_d_frame, "d", x=-0.075, y=1.03)

    # Align Panel D's full visual extent with the panels above. The heatmap's
    # two-line y-tick labels and the colorbar label are wider than the bar
    # panels' numeric ticks, so align by measured tight-bbox overhang rather
    # than by spine: pin D's outer-left to A/B and its outer-right to A/C,
    # which shifts the heatmap inward and lets it compress to fit.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    inv_fig = fig.transFigure.inverted()
    bb_top = ax_a.get_tightbbox(renderer).transformed(inv_fig)
    bb_heat = ax_d.get_tightbbox(renderer).transformed(inv_fig)
    bb_cbar = cax_d.get_tightbbox(renderer).transformed(inv_fig)
    pos_heat = ax_d.get_position()
    pos_cbar = cax_d.get_position()
    left_pad = pos_heat.x0 - bb_heat.x0     # y-title + tick-label overhang
    right_pad = bb_cbar.x1 - pos_cbar.x1    # colorbar tick-label + label overhang
    new_heat_x0 = bb_top.x0 + left_pad
    new_cbar_x0 = bb_top.x1 - right_pad - cbar_w
    new_heat_x1 = new_cbar_x0 - cbar_gap
    ax_d.set_position([new_heat_x0, pos_heat.y0, new_heat_x1 - new_heat_x0, pos_heat.height])
    cax_d.set_position([new_cbar_x0, pos_cbar.y0, cbar_w, pos_cbar.height])

    save_triplet(fig, "fig_sensitivity_boundary_atlas", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_9_boundary_atlas()


In [ ]:
# Retired standalone figure: joint SMR and absorption-chiller capital-cost frontier (fig_capital_cost_frontier)
# This content is now Figure 9, Panel D. Keep the function only for optional reuse.
def figure_10_capital_cost_frontier() -> None:
    fig, ax_h = plt.subplots(figsize=(BODY_W, 3.80))
    fig.subplots_adjust(left=0.16, right=0.90, top=0.98, bottom=0.20)

    s5 = df[df.group == "s5_feasibility_2d"].copy()
    smr_order = ["NOAK", "Low_Mid", "ATB_Mid", "High_Mid", "FOAK"]
    abs_order = ["Bare_Low", "Mid_Low", "Baseline", "Mid_High", "Turnkey_High"]
    smr_vals = [2250, 5000, 7615, 11000, 14700]
    abs_vals = [450, 600, 750, 900, 1200]
    piv = s5.pivot_table(index="smr_capex_tag", columns="absorption_capex_tag", values="premium_pct")
    piv = piv.reindex(smr_order)[abs_order]
    data = piv.values

    cmap = LinearSegmentedColormap.from_list(
        "hrp_blue", [PALETTE["fill_blue"], "#FFFFFF"], N=256
    )
    im = ax_h.imshow(data, cmap=cmap, vmin=float(data.min()), vmax=0.0, aspect="auto")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax_h.text(j, i, f"{data[i, j]:+.0f}%", ha="center", va="center",
                      fontsize=8.0, color="#222222")
    ax_h.set_xticks(np.arange(len(abs_order)))
    ax_h.set_xticklabels([f"{v}" for v in abs_vals], rotation=35, ha="right")
    ax_h.set_yticks(np.arange(len(smr_order)))
    ax_h.set_yticklabels([f"{t.replace('_', '-')}\n\\${v:,}" for t, v in zip(smr_order, smr_vals)])
    ax_h.set_xlabel(r"Absorption CAPEX (\$/kW$_\mathrm{c}$)")
    ax_h.set_ylabel(r"SMR CAPEX trajectory (\$/kW$_\mathrm{e}$)")
    ax_h.set_xticks(np.arange(-.5, len(abs_order), 1), minor=True)
    ax_h.set_yticks(np.arange(-.5, len(smr_order), 1), minor=True)
    ax_h.grid(which="minor", color=BLOCK_EDGE, linestyle="-", linewidth=1.0)
    ax_h.tick_params(which="minor", bottom=False, left=False)
    cb = fig.colorbar(im, ax=ax_h, fraction=0.046, pad=0.025)
    cb.set_label("Case 2 grid-cost margin (%)")

    save_triplet(fig, "fig_capital_cost_frontier", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(figure_10_capital_cost_frontier)


In [ ]:
# Figure 10: Policy crossover and global sensitivity ranking (fig_policy_sensitivity_summary)
# Panels A-B: carbon-price crossover and ranked one-variable sensitivity.
def figure_10_policy_sensitivity_summary() -> None:
    fig = plt.figure(figsize=(BODY_W, 6.05))
    gs = fig.add_gridspec(
        2, 1,
        height_ratios=[1.10, 1.28],
        left=0.15, right=0.98, top=0.98, bottom=0.10,
        hspace=0.22,
    )
    ax_c = fig.add_subplot(gs[0, 0])
    ax_t = fig.add_subplot(gs[1, 0])

    # a) Carbon-price crossover.
    s6 = df[df.group == "s6_carbon_price"].copy().sort_values(["case_id", "carbon_price_usd_per_tco2"])
    x_extrap = np.linspace(0, 250, 256)
    crossings: dict[int, float] = {}
    fits: dict[int, tuple[float, float]] = {}
    panel_a_line_colors = FIGURE2_LINE_COLORS.copy()
    panel_a_line_colors[1] = PALETTE["accent_sky"]
    case0_at_x = None
    for cid in (0, 1, 2, 3):
        sub = s6[s6.case_id == cid]
        x = sub.carbon_price_usd_per_tco2.values
        y = sub.tac_usd_per_yr.values / 1e6
        slope = (y[-1] - y[0]) / (x[-1] - x[0])
        intercept = y[0]
        fits[cid] = (intercept, slope)
        y_extrap = intercept + slope * x_extrap
        line_color = panel_a_line_colors[cid]
        ax_c.plot(x_extrap, y_extrap, color=line_color, linewidth=LINE_PLOT_LW, zorder=2)
        ax_c.scatter(x, y, marker=CASE_MARKER[cid], s=LINE_MARKER_AREA,
                     facecolor=MAIN_FILLS[3], edgecolor=BLOCK_EDGE,
                     linewidth=LINE_MARKER_EDGE_WIDTH, zorder=4)
        if cid == 0:
            case0_at_x = (intercept, slope)
        else:
            c0_int, c0_slope = case0_at_x
            denom = slope - c0_slope
            if abs(denom) > 1e-9:
                cross_price = (c0_int - intercept) / denom
                if 0 < cross_price < 260:
                    crossings[cid] = cross_price

    nuclear_cross = np.mean([crossings[c] for c in (1, 2) if c in crossings])
    ngcc_cross = crossings.get(3, 250)
    ax_c.axvspan(nuclear_cross, ngcc_cross, color=PALETTE["fill_blue"], alpha=0.16, lw=0)
    ax_c.axvspan(ngcc_cross, 250, color=PALETTE["fill_salmon"], alpha=0.13, lw=0)
    ax_c.axvline(nuclear_cross, color=PALETTE["stroke_navy"], ls=":", lw=1.0)
    ax_c.text(nuclear_cross - 4, 230, f"Nuclear\n~\\${nuclear_cross:.0f}/tCO$_2$",
              fontsize=TEXT_SIZE, ha="right", va="top", color=PALETTE["stroke_navy"],
              linespacing=0.95)
    ax_c.text((nuclear_cross + ngcc_cross) / 2 - 4, 76, "Nuclear\nless than\ngrid",
              fontsize=TEXT_SIZE, ha="center", va="center", color=PALETTE["stroke_navy"],
              linespacing=1.18)
    ax_c.text((ngcc_cross + 250) / 2 + 4, 70, "Nuclear\nless than\nGrid and NGCC",
              fontsize=TEXT_SIZE, ha="center", va="center", color=PALETTE["stroke_clay"],
              linespacing=1.20)
    if 3 in crossings:
        ax_c.axvline(crossings[3], color="#555555", ls=":", lw=0.9)
        ax_c.text(crossings[3] + 4, 230, f"NGCC\n~\\${crossings[3]:.0f}/tCO$_2$",
                  fontsize=TEXT_SIZE, ha="left", va="top", color="#555555",
                  linespacing=0.95)

    direct_labels = {
        0: "C0 grid",
        1: "C1 nuclear",
        2: "C2 absorption",
        3: "C3 NGCC",
    }
    label_offsets = {0: -7.5, 1: 9.0, 2: -9.5, 3: 8.0}
    for cid, text in direct_labels.items():
        intercept, slope = fits[cid]
        y_end = intercept + slope * 250
        ax_c.text(253.5, y_end + label_offsets[cid], text,
                  ha="left", va="center", fontsize=TEXT_SIZE, color=panel_a_line_colors[cid])
    ax_c.set_xlabel(r"Carbon price (\$/tCO$_2$)")
    ax_c.set_ylabel("TAC (M\$/yr)")
    ax_c.set_xlim(0, 272)
    ax_c.set_ylim(40, 235)
    ax_c.set_axisbelow(True)
    ax_c.grid(**GRID_KW)
    # Figure 10, Panel A: TAC extrapolated against carbon price.
    add_panel_label(ax_c, "a", x=-0.12, y=1.05)

    # b) Ranked tornado sensitivity.
    def hrp_percent(run_id: str) -> float:
        return float(df.loc[df.run_id == run_id, "heat_recovery_premium"].iloc[0] * 100.0)

    baseline = hrp_percent("case2_ATB_Mid")
    axes_def = [
        ("SMR CAPEX", "case2_FOAK", "case2_NOAK"),
        ("Market year", "case2_year2024", "case2_year2022"),
        ("WACC", "case2_wacc_100", "case2_wacc_50"),
        ("Carbon price", "case2_co2_0", "case2_co2_100"),
        ("DC size", "case2_load_x050", "case2_load_x300"),
        ("Absorption CAPEX", "case2_smr_ATB_Mid_abs_Turnkey_High", "case2_smr_ATB_Mid_abs_Bare_Low"),
        ("PUE", "case2_pue150", "case2_pue110"),
        ("BESS on/off", "case2_bess_off", "case2_bess_on"),
    ]
    axis_spans = []
    for label, run_a, run_b in axes_def:
        va = hrp_percent(run_a)
        vb = hrp_percent(run_b)
        lo, hi = min(va, vb), max(va, vb)
        axis_spans.append((label, lo, hi, hi - lo))
    axis_spans.sort(key=lambda row: row[3], reverse=True)

    labels = [row[0] for row in axis_spans]
    lows = [row[1] for row in axis_spans]
    highs = [row[2] for row in axis_spans]
    spans = [row[3] for row in axis_spans]
    ypos = np.arange(len(axis_spans))[::-1]
    segment_lengths = []
    for lo, hi in zip(lows, highs):
        left = min(lo, baseline)
        right = max(hi, baseline)
        if left < baseline:
            segment_lengths.append(baseline - left)
        if right > baseline:
            segment_lengths.append(right - baseline)
    max_segment_length = max(segment_lengths) if segment_lengths else 0.0

    def gamma_from_length(length: float, max_gamma: float = 3.5) -> float:
        if max_segment_length <= 0:
            return 1.0
        return 1.0 + (max_gamma - 1.0) * (length / max_segment_length)
    worse_color = MAIN_FILLS[0]
    better_color = MAIN_FILLS[1]
    for y, lo, hi in zip(ypos, lows, highs):
        left = min(lo, baseline)
        right = max(hi, baseline)
        if left < baseline:
            draw_horizontal_gradient_bar(
                ax_t, y, left, baseline, 0.62, worse_color,
                white_at="right", gamma=gamma_from_length(baseline - left),
            )
        if right > baseline:
            draw_horizontal_gradient_bar(
                ax_t, y, baseline, right, 0.62, better_color,
                white_at="left", gamma=gamma_from_length(right - baseline),
            )
        annotate_horizontal_endpoint(
            ax_t, lo, y, f"{lo:.0f}%", side="left", color=MAIN_STROKES[0]
        )
        annotate_horizontal_endpoint(
            ax_t, hi, y, f"{hi:.0f}%", side="right", color=MAIN_STROKES[1]
        )
    ax_t.axvline(baseline, color="#333333", lw=1.0, ls="--", zorder=8)
    ax_t.annotate(
        f"ATB-Mid baseline ({baseline:.0f}%)",
        xy=(baseline, len(axis_spans) - 0.55), xytext=(POINT_LABEL_OFFSET_PT, 0),
        textcoords="offset points",
        ha="left", va="center", fontsize=8, color="#333333", zorder=9,
    )
    ax_t.set_yticks(ypos)
    ax_t.set_yticklabels([f"{label}\n(span {span:.0f} pp)" for label, span in zip(labels, spans)])
    ax_t.set_xlabel("Case 2 grid-cost margin (%)")
    ax_t.set_xlim(min(lows) - 90, max(highs) + 40)
    ax_t.set_ylim(-1.0, len(axis_spans) - 0.4)
    ax_t.set_axisbelow(True)
    ax_t.grid(axis="x", alpha=0.25, linestyle="--", linewidth=0.5)
    ax_t.tick_params(direction="out", length=3)
    for spine in ("top", "right"):
        ax_t.spines[spine].set_visible(False)
    handles = [
        Patch(facecolor=worse_color, edgecolor="white", label="more negative than baseline"),
        Patch(facecolor=better_color, edgecolor="white", label="less negative than baseline"),
    ]
    ax_t.legend(handles=handles, loc="lower left", frameon=False, fontsize=8,
                bbox_to_anchor=(0.0, 0.02))
    # Figure 10, Panel B: ranked sensitivity of Case 2 grid-cost margin.
    add_panel_label(ax_t, "b", x=-0.12, y=1.03)

    save_triplet(fig, "fig_policy_sensitivity_summary", str(FIGURES))
    display(fig)
    plt.close(fig)

figure_10_policy_sensitivity_summary()


## Unused / Supplemental Figure Cells

The cells below are not referenced by the current compiled `main.pdf`. They stay after Figure 1-Figure 10 so future agents do not confuse exploratory or retired figures with the manuscript sequence. They are opt-in only: set `RENDER_UNUSED_FIGURES = True` in the setup cell before running them.


In [ ]:
# Unused figure (not in current main.pdf): fig5_s1_pue
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig5_s1_pue() -> None:
    s1 = df[df.group == "s1_pue"].sort_values(["case_id", "pue"]).copy()

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.45))
    two_case_style = {
        1: (MAIN_STROKES[0], MAIN_FILLS[0]),
        2: (MAIN_STROKES[1], MAIN_FILLS[1]),
    }
    for cid in (1, 2):
        sub = s1[s1.case_id == cid]
        line_color, fill_color = two_case_style[cid]
        ax.plot(sub.pue, sub.premium_pct,
                marker=CASE_MARKER[cid], markersize=6,
                color=line_color, linewidth=1.5,
                markerfacecolor=fill_color,
                markeredgecolor=line_color, markeredgewidth=0.9,
                label={1: "C1 nuclear, no recovery",
                       2: "C2 nuclear + absorption"}[cid])

    ax.axhline(0, label="Margin = 0", **ZERO_LINE_KW)
    ax.set_xlabel("PUE")
    ax.set_ylabel("Grid-cost margin (%)")
    ax.set_xticks([1.10, 1.30, 1.50])
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="center right", bbox_to_anchor=(0.98, 0.60),
              frameon=False, fontsize=8, handlelength=1.4,
              borderaxespad=0.0, labelspacing=0.35)
    fig.tight_layout()
    save_triplet(fig, "fig5_s1_pue", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig5_s1_pue)

In [ ]:
# Unused figure (not in current main.pdf): fig_c1c2_delta
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig_c1c2_delta() -> None:
    pue_levels = [1.10, 1.30, 1.50]
    pue_gap = []
    s1 = df[df.group == "s1_pue"].copy()
    for pue in pue_levels:
        c1 = s1[(s1.case_id == 1) & (s1.pue.round(2) == pue)].heat_recovery_premium.iloc[0] * 100
        c2 = s1[(s1.case_id == 2) & (s1.pue.round(2) == pue)].heat_recovery_premium.iloc[0] * 100
        pue_gap.append(c2 - c1)

    co2_levels = [0.0, 50.0, 100.0]
    co2_gap = []
    s6 = df[df.group == "s6_carbon_price"].copy()
    for price in co2_levels:
        c1 = s6[(s6.case_id == 1) & (s6.carbon_price_usd_per_tco2 == price)].heat_recovery_premium.iloc[0] * 100
        c2 = s6[(s6.case_id == 2) & (s6.carbon_price_usd_per_tco2 == price)].heat_recovery_premium.iloc[0] * 100
        co2_gap.append(c2 - c1)

    fig, (ax_l, ax_r) = plt.subplots(1, 2, figsize=(BODY_W, 3.0), sharey=True)

    x_l = np.arange(len(pue_levels))
    ax_l.axhline(0.0, **ZERO_LINE_KW)
    ax_l.plot(
        x_l, pue_gap, "-", color=MAIN_STROKES[0], marker="o", markersize=5,
        markerfacecolor=MAIN_STROKES[0], markeredgecolor="white", markeredgewidth=0.6,
        zorder=3,
    )
    for x, y in zip(x_l, pue_gap):
        ax_l.annotate(f"{y:+.1f}", (x, y), textcoords="offset points",
                      xytext=(0, -11), ha="center", va="top", fontsize=8,
                      color=MAIN_STROKES[0])
    ax_l.set_xticks(x_l)
    ax_l.set_xticklabels([f"{p:.2f}" for p in pue_levels])
    ax_l.set_xlabel("Data-center PUE")
    ax_l.set_ylabel("Case 2 - Case 1 margin gap (pp)")
    ax_l.set_xlim(-0.4, len(pue_levels) - 0.6)
    add_panel_label(ax_l, "a", x=-0.12, y=1.02)

    x_r = np.arange(len(co2_levels))
    ax_r.axhline(0.0, **ZERO_LINE_KW)
    ax_r.plot(
        x_r, co2_gap, "-", color=MAIN_STROKES[1], marker="s", markersize=5,
        markerfacecolor=MAIN_STROKES[1], markeredgecolor="white", markeredgewidth=0.6,
        zorder=3,
    )
    for x, y in zip(x_r, co2_gap):
        ax_r.annotate(f"{y:+.1f}", (x, y), textcoords="offset points",
                      xytext=(0, -11), ha="center", va="top", fontsize=8,
                      color=MAIN_STROKES[1])
    ax_r.set_xticks(x_r)
    ax_r.set_xticklabels([f"{int(c)}" for c in co2_levels])
    ax_r.set_xlabel(r"Carbon price (\$/tCO$_2$)")
    ax_r.set_xlim(-0.4, len(co2_levels) - 0.6)
    add_panel_label(ax_r, "b", x=-0.12, y=1.02)

    ymin = min(min(pue_gap), min(co2_gap))
    for ax in (ax_l, ax_r):
        ax.set_ylim(ymin - 2.5, 2.0)
        ax.set_axisbelow(True)
        ax.grid(**GRID_KW)
        ax.tick_params(direction="out", length=3)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    fig.tight_layout()
    save_triplet(fig, "fig_c1c2_delta", str(FIGURES))
    display(fig)
    plt.close(fig)


maybe_render_unused_figure(fig_c1c2_delta)

In [ ]:
# Unused figure (not in current main.pdf): fig6_s2_year_regime
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig6_s2_year_regime() -> None:
    s2 = df[df.group == "s2_price"].copy().sort_values(["year", "case_id"])

    years = sorted(s2.year.unique())
    cases = sorted(s2.case_id.unique())
    x = np.arange(len(years))
    bar_w = 0.18

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.95))
    # This unused year-regime plot uses a local shifted case order.
    # gray first, then the standard blue, salmon/pink, and yellow sequence.
    fig9_fill = {
        0: MAIN_FILLS[3],
        1: MAIN_FILLS[0],
        2: MAIN_FILLS[1],
        3: MAIN_FILLS[2],
    }
    for i, cid in enumerate(cases):
        vals = [
            s2[(s2.year == y) & (s2.case_id == cid)].premium_pct.iloc[0]
            for y in years
        ]
        offset = (i - (len(cases) - 1) / 2) * bar_w
        ax.bar(x + offset, vals, width=bar_w,
               color=fig9_fill[cid], edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
               label=CASE_LABEL[cid])

    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x)
    ax.set_xticklabels([str(y) for y in years])
    ax.set_xlabel("ERCOT year")
    ax.set_ylabel("Grid-cost margin (%)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="lower left", bbox_to_anchor=(0.02, 0.04), ncol=2,
              frameon=False, fontsize=8, columnspacing=0.8, handlelength=1.1,
              borderaxespad=0.0)
    fig.tight_layout()
    save_triplet(fig, "fig6_s2_year_regime", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig6_s2_year_regime)

In [ ]:
# Unused figure (not in current main.pdf): fig7_s3_bess
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig7_s3_bess() -> None:
    s3 = df[df.group == "s3_battery"].copy()
    # Pivot to wide form: TAC_off, TAC_on per case
    wide = s3.pivot_table(index="case_id", columns="bess_applied",
                          values="tac_usd_per_yr")
    wide["delta_M"] = (wide.get(True, np.nan) - wide.get(False, np.nan)) / 1e6
    # CO2 same way
    co2_wide = s3.pivot_table(index="case_id", columns="bess_applied",
                              values="co2_annual_tonnes")

    # $/tCO2 avoided vs Case 0 baseline (BESS off)
    tac_off = wide.get(False, np.nan)
    co2_off = co2_wide.get(False, np.nan)
    tac_case0 = tac_off.loc[0]
    co2_case0 = co2_off.loc[0]
    abate = []
    for cid in tac_off.index:
        d_tac = tac_off.loc[cid] - tac_case0
        d_co2 = co2_case0 - co2_off.loc[cid]
        abate.append(d_tac / d_co2 if abs(d_co2) > 1e-3 else np.nan)
    wide["abate_usd_per_tco2"] = abate

    cases = wide.index.tolist()
    labels = [f"C{c}" for c in cases]
    fill_colors = [CASE_FILL[c] for c in cases]
    edge_colors = [CASE_LINE[c] for c in cases]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(SINGLE_W, 3.35))

    # Panel a — ΔTAC with BESS
    ax1.bar(labels, wide.delta_M.values, color=fill_colors,
            edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    ax1.axhline(0, **ZERO_LINE_KW)
    ax1.set_ylabel(r"$\Delta$TAC with BESS (M\$/yr)")
    ax1.set_xlabel("Case")
    ax1.set_axisbelow(True)
    ax1.grid(**GRID_KW)
    add_panel_label(ax1, "a", x=-0.10, y=1.02)

    # Panel b — $/tCO2 abated
    ax2.bar(labels, wide.abate_usd_per_tco2.values, color=fill_colors,
            edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    ax2.axhline(0, **ZERO_LINE_KW)
    ax2.set_ylabel(r"Carbon abatement (\$/tCO$_2$)")
    ax2.set_xlabel("Case")
    ax2.set_axisbelow(True)
    ax2.grid(**GRID_KW)
    add_panel_label(ax2, "b", x=-0.10, y=1.02)

    fig.tight_layout()
    save_triplet(fig, "fig7_s3_bess", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig7_s3_bess)

In [ ]:
# Unused figure (not in current main.pdf): fig8_s4_capex_1d
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig8_s4_capex_1d() -> None:
    s4 = df[df.group == "s4_capex"].copy()
    scen_order = ["FOAK", "ATB_Mid", "NOAK"]
    x = np.arange(len(scen_order))
    bar_w = 0.36

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.45))
    for i, cid in enumerate((1, 2)):
        vals = [
            s4[(s4.reactor_scenario == s) & (s4.case_id == cid)].premium_pct.iloc[0]
            for s in scen_order
        ]
        offset = (i - 0.5) * bar_w
        fill_color = MAIN_FILLS[i]
        ax.bar(x + offset, vals, width=bar_w,
               color=fill_color, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW,
               label=CASE_LABEL[cid])
        for xi, v in zip(x + offset, vals):
            if v < -40:
                y_lab = v + 14
                va = "bottom"
            elif v < 0:
                y_lab = v - 7
                va = "top"
            else:
                y_lab = v + 4
                va = "bottom"
            ax.text(xi, y_lab, f"{v:+.0f}%",
                    ha="center", va=va, fontsize=8, color="#000000")

    ax.axhline(0, **ZERO_LINE_KW)
    ax.set_xticks(x)
    ax.set_xticklabels(["FOAK\n$14,700", "ATB-Mid\n$7,615", "NOAK\n$2,250"], fontsize=8)
    ax.set_xlabel(r"BWRX-300 OCC (\$/kW$_\mathrm{e}$)")
    ax.set_ylabel("Grid-cost margin (%)")
    ax.set_axisbelow(True)
    ax.grid(**GRID_KW)
    ax.legend(loc="lower right", frameon=False, fontsize=8)
    ax.set_ylim(-530, 25)
    fig.tight_layout()
    save_triplet(fig, "fig8_s4_capex_1d", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig8_s4_capex_1d)

In [ ]:
# Unused figure (not in current main.pdf): fig9_s5_feasibility_2d
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig9_s5_feasibility_2d() -> None:
    s5 = df[df.group == "s5_feasibility_2d"].copy()
    smr_order = ["NOAK", "Low_Mid", "ATB_Mid", "High_Mid", "FOAK"]
    abs_order = ["Bare_Low", "Mid_Low", "Baseline", "Mid_High", "Turnkey_High"]
    smr_vals = [2250, 5000, 7615, 11000, 14700]
    abs_vals = [450, 600, 750, 900, 1200]

    piv = s5.pivot_table(index="smr_capex_tag", columns="absorption_capex_tag",
                         values="premium_pct")
    piv = piv.reindex(smr_order)[abs_order]
    data = piv.values  # shape (5, 5)

    # Custom diverging colormap centered on 0
    cmap = LinearSegmentedColormap.from_list(
        "sf_div",
        [PALETTE["fill_blue"], "#FFFFFF", PALETTE["fill_salmon"]],
        N=256,
    )
    # All data are negative in this study; force symmetric centering so the
    # Premium=0 contour reads correctly if any future tweak pushes a cell
    # above zero.
    vmax = max(abs(data.min()), abs(data.max()), 30)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(BODY_W, 4.6))
    im = ax.imshow(data, cmap=cmap, norm=norm, aspect="auto")

    # Cell value labels
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            ax.text(j, i, f"{v:+.0f}%", ha="center", va="center", fontsize=8,
                    color="#222222")

    # Margin = 0 contour
    cs = ax.contour(
        np.arange(data.shape[1]),
        np.arange(data.shape[0]),
        data,
        levels=[0.0],
        colors=[PALETTE["stroke_navy"]],
        linewidths=1.4,
        linestyles="--",
    )
    try:
        ax.clabel(cs, fmt={0.0: "Margin = 0"}, fontsize=8, inline=True)
    except (ValueError, IndexError):
        pass  # contour empty (no zero-crossing in grid)

    # Tick labels: combine tag + $ value
    smr_tick = [f"{t}\n${v:,}" for t, v in zip(smr_order, smr_vals)]
    abs_tick = [f"{t}\n${v}" for t, v in zip(abs_order, abs_vals)]
    ax.set_xticks(np.arange(len(abs_order)))
    ax.set_xticklabels(abs_tick, fontsize=8)
    ax.set_yticks(np.arange(len(smr_order)))
    ax.set_yticklabels(smr_tick, fontsize=8)
    ax.set_xlabel(r"Absorption CAPEX (\$/kW$_\mathrm{c}$)")
    ax.set_ylabel(r"SMR CAPEX (\$/kW$_\mathrm{e}$)")
    # Minor-grid separators between cells (NPPH2 P5)
    ax.set_xticks(np.arange(-0.5, len(abs_order), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(smr_order), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", length=0)

    # Anchor markers — placed without inline labels (overlap with cell values).
    # Anchor identity carried by marker shape + color in the legend strip below.
    grid_path = PROJECT_ROOT / "config" / "capex_grid_s5.yaml"
    with grid_path.open() as f:
        grid = yaml.safe_load(f)
    anchor_style = {
        "red_circle":    (MAIN_STROKES[0], "o"),
        "green_circle":  (MAIN_STROKES[1], "s"),
        "yellow_circle": (MAIN_STROKES[2], "D"),
        "blue_circle":   (MAIN_STROKES[3], "^"),
        "purple_circle": (MAIN_STROKES[0], "v"),
    }
    legend_handles: list[Line2D] = []
    anchor_offsets = {
        "red_circle": (-0.24, 0.22),
        "green_circle": (0.24, -0.22),
        "yellow_circle": (0.24, 0.22),
        "blue_circle": (-0.24, -0.22),
        "purple_circle": (0.24, 0.22),
    }
    for a in grid["anchors"]:
        i = smr_order.index(a["smr_tag"])
        j = abs_order.index(a["absorption_tag"])
        color, marker = anchor_style.get(a["marker"], (MAIN_STROKES[3], "x"))
        dx, dy = anchor_offsets.get(a["marker"], (0.24, 0.22))
        ax.scatter(j + dx, i + dy, marker=marker, s=82,
                   facecolor=color, edgecolor=BLOCK_EDGE, linewidth=0.9,
                   zorder=5)
        # Short label for the legend strip (drop the parenthetical aside)
        short = a["label"].split("(")[0].strip()
        legend_handles.append(
            Line2D([0], [0], marker=marker, color="none",
                   markerfacecolor=color, markeredgecolor=BLOCK_EDGE,
                   markersize=7, label=short)
        )

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Grid-cost margin (%)", fontsize=8)
    cbar.ax.tick_params(labelsize=8)

    # Anchor legend below the heatmap
    ax.legend(handles=legend_handles, loc="upper center",
              bbox_to_anchor=(0.5, -0.16), ncol=3, frameon=False,
              fontsize=7.5, handlelength=0.9, columnspacing=0.75,
              handletextpad=0.35, borderaxespad=0.0)

    fig.tight_layout()
    save_triplet(fig, "fig9_s5_feasibility_2d", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig9_s5_feasibility_2d)

In [ ]:
# Unused figure (not in current main.pdf): fig10_s6_carbon_price
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig10_s6_carbon_price() -> None:
    s6 = df[df.group == "s6_carbon_price"].copy()
    s6 = s6.sort_values(["case_id", "carbon_price_usd_per_tco2"])

    fig, ax = plt.subplots(figsize=(SINGLE_W, 2.85))

    # Plot the 3 actual data points per case, then extrapolate linearly to
    # carbon = $250 so the crossovers visualise on-canvas.
    x_extrap = np.linspace(0, 250, 256)
    crossings: dict[int, float] = {}
    case0_at_x = None

    for cid in (0, 1, 2, 3):
        sub = s6[s6.case_id == cid]
        x = sub.carbon_price_usd_per_tco2.values
        y = sub.tac_usd_per_yr.values / 1e6
        # Slope is exact because TAC is linear in carbon price by construction.
        slope = (y[-1] - y[0]) / (x[-1] - x[0])
        intercept = y[0]
        y_extrap = intercept + slope * x_extrap

        ax.plot(x_extrap, y_extrap,
                color=CASE_LINE[cid], linewidth=1.3, zorder=2,
                label=CASE_LABEL[cid])
        ax.scatter(x, y, marker=CASE_MARKER[cid], s=42,
                   facecolor=CASE_FILL[cid], edgecolor=CASE_LINE[cid],
                   linewidth=0.9, zorder=4)

        if cid == 0:
            case0_at_x = (intercept, slope)
        else:
            c0_int, c0_slope = case0_at_x
            denom = (slope - c0_slope)
            if abs(denom) > 1e-9:
                cross_price = (c0_int - intercept) / denom
                if 0 < cross_price < 260:
                    crossings[cid] = cross_price

    # Group near-identical Case 1 / Case 2 crossovers into one annotation
    # (they sit within ~$1/tCO2 because the cogen Premium ≈ pure-nuclear).
    cid_groups: list[tuple[list[int], float]] = []
    used: set[int] = set()
    for cid in (1, 2, 3):
        if cid not in crossings or cid in used:
            continue
        partners = [cid]
        used.add(cid)
        for other in (1, 2, 3):
            if other in crossings and other not in used and abs(
                crossings[other] - crossings[cid]
            ) < 3.0:
                partners.append(other)
                used.add(other)
        cid_groups.append(
            (sorted(partners), float(np.mean([crossings[c] for c in partners])))
        )

    for cids, p_cross in cid_groups:
        c0_int, c0_slope = case0_at_x
        tac_cross = c0_int + c0_slope * p_cross
        primary = cids[-1]  # darker hue if grouped (C2 > C1)
        ax.axvline(p_cross, color=CASE_LINE[primary],
                   linestyle=":", linewidth=1.0, alpha=0.85, zorder=1)
        label = "/".join(f"C{c}" for c in cids) + f": \\${p_cross:.0f}"
        ax.annotate(
            label,
            xy=(p_cross, tac_cross),
            xytext=(7, 9 if 2 in cids else -14),
            textcoords="offset points",
            fontsize=8, color=CASE_LINE[primary],
            ha="left", va="center",
        )

    # Shade the "nuclear cheaper than grid" region (right of last crossover).
    if crossings:
        p_max = max(crossings.values())
        ax.axvspan(p_max, 250, alpha=0.07,
                   color=PALETTE["stroke_teal"], zorder=0)
        # Tag anchored in axes coords so it never collides with data lines.
        x_norm = ((p_max + 250) / 2) / 250
        ax.text(x_norm, 0.965, "Nuclear $<$ grid",
                transform=ax.transAxes, ha="center", va="top",
                fontsize=8, color=PALETTE["stroke_teal"], style="italic")

    ax.set_xlim(-5, 250)
    ax.set_xlabel(r"Carbon price (\$/tCO$_2$)")
    ax.set_ylabel(r"TAC (M\$/yr)")
    ax.set_axisbelow(True)
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.5)

    # Legend above the data area, using the page whitespace and preserving x-axis labels.
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=4,
              frameon=False, fontsize=7.3, handlelength=1.25,
              columnspacing=0.75, handletextpad=0.35, borderaxespad=0.0)

    fig.tight_layout()
    save_triplet(fig, "fig10_s6_carbon_price", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig10_s6_carbon_price)

In [ ]:
# Unused figure (not in current main.pdf): fig_si_kpi_panel
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig11_kpi_panel() -> None:
    base = df[df.group == "main_baseline"].sort_values("case_id").reset_index(drop=True)
    cases = base.case_id.astype(int).values
    labels = [f"C{c}" for c in cases]
    colors = [CASE_FILL[c] for c in cases]
    edge_colors = [CASE_LINE[c] for c in cases]

    fig, axes = plt.subplots(2, 2, figsize=(BODY_W, 5.0))

    # (a) LCOE — $/MWh_e delivered
    lcoe = base.lcoe_usd_per_mwh_e.values
    axes[0, 0].bar(labels, lcoe, color=colors, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    for x, v in zip(labels, lcoe):
        axes[0, 0].text(x, v + 5, f"{v:.0f}", ha="center", va="bottom",
                        fontsize=8, color="#000000")
    axes[0, 0].set_ylabel(r"LCOE (\$/MWh$_\mathrm{e}$)")
    axes[0, 0].set_axisbelow(True)
    axes[0, 0].grid(**GRID_KW)
    add_panel_label(axes[0, 0], "a")

    # (b) EPBT — years (Case 0 has no on-site plant → empty bar)
    epbt = base.epbt_years.fillna(0).values
    bars = axes[0, 1].bar(labels, epbt, color=colors, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    for x, v, raw in zip(labels, epbt, base.epbt_years.values):
        if np.isnan(raw):
            axes[0, 1].text(x, 0.02, "n/a", ha="center", va="bottom",
                            fontsize=8, color="#666")
        else:
            axes[0, 1].text(x, v + 0.02, f"{v:.2f}", ha="center", va="bottom",
                            fontsize=8, color="#000000")
    axes[0, 1].set_ylabel(r"EPBT (years)")
    axes[0, 1].set_axisbelow(True)
    axes[0, 1].grid(**GRID_KW)
    add_panel_label(axes[0, 1], "b")

    # (c) Water footprint — total L/MWh_e (v2.7 3-tier; subpanel uses total)
    water = base.water_total_l_per_mwh_e.values
    axes[1, 0].bar(labels, water, color=colors, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    axes[1, 0].set_yscale("log")
    for x, v in zip(labels, water):
        axes[1, 0].text(x, v * 1.18, f"{v:,.0f}", ha="center", va="bottom",
                        fontsize=8, color="#000000")
    axes[1, 0].set_ylabel(r"Water (L/MWh$_\mathrm{e}$, log)")
    axes[1, 0].set_axisbelow(True)
    axes[1, 0].grid(axis="y", alpha=0.3, linestyle="--", linewidth=0.5, which="both")
    add_panel_label(axes[1, 0], "c")

    # (d) Carbon abatement cost — $/tCO2 avoided (Cases 0,3 → n/a)
    abate = base.carbon_abatement_cost_usd_per_tco2.fillna(0).values
    raw_abate = base.carbon_abatement_cost_usd_per_tco2.values
    axes[1, 1].bar(labels, abate, color=colors, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW)
    for x, v, raw in zip(labels, abate, raw_abate):
        if np.isnan(raw):
            axes[1, 1].text(x, 5, "n/a", ha="center", va="bottom",
                            fontsize=8, color="#666")
        else:
            axes[1, 1].text(x, v + 5, f"\\${v:.0f}", ha="center", va="bottom",
                            fontsize=8, color="#000000")
    axes[1, 1].set_ylabel(r"Abatement (\$/tCO$_2$)")
    axes[1, 1].set_axisbelow(True)
    axes[1, 1].grid(**GRID_KW)
    add_panel_label(axes[1, 1], "d")

    for ax in axes.flat:
        ax.set_xlabel("")

    fig.tight_layout()
    save_triplet(fig, "fig_si_kpi_panel", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig11_kpi_panel)

In [ ]:
# Unused figure (not in current main.pdf): fig11_s7_wacc_leverage
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig11_s7_wacc_leverage() -> None:
    # S7 rows
    s7 = df[df.group == "s7_wacc"].sort_values("wacc_effective")
    # S4 rows (Case 2 only — match the S7 Case 2 lens)
    s4 = (
        df[(df.group == "s4_capex") & (df.case_id == 2)]
        .copy()
        .sort_values(
            "reactor_scenario", key=lambda s: s.map({"FOAK": 0, "ATB_Mid": 1, "NOAK": 2})
        )
    )

    if s7.empty or s4.empty:
        print("[fig11] missing s7 or s4 rows; skipping")
        return

    # Premium points (percentage)
    wacc_pts = list(zip(s7.wacc_effective.values, s7.premium_pct.values))
    capex_pts = list(zip(s4.reactor_scenario.values, s4.premium_pct.values))

    # ΔPremium spans
    wacc_min = min(p for _, p in wacc_pts)
    wacc_max = max(p for _, p in wacc_pts)
    capex_min = min(p for _, p in capex_pts)
    capex_max = max(p for _, p in capex_pts)

    fig, axes = plt.subplots(2, 1, figsize=(SINGLE_W, 3.4), sharey=True)

    # Left: WACC sweep (5% / 6.7% / 10%)
    waccs = [w * 100 for w, _ in wacc_pts]
    prems_wacc = [p for _, p in wacc_pts]
    colors_wacc = [
        MAIN_FILLS[0] if p > prems_wacc[1] else
        (MAIN_FILLS[1] if p < prems_wacc[1] else MAIN_FILLS[3])
        for p in prems_wacc
    ]
    bars_l = axes[0].bar(
        [f"{w:.1f}%" for w in waccs], prems_wacc,
        color=colors_wacc, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=0.6,
    )
    for b, v in zip(bars_l, prems_wacc):
        axes[0].text(
            b.get_x() + b.get_width() / 2,
            v + (8 if v < 0 else -8),
            f"{v:.0f}%",
            ha="center", va="center", fontsize=8, color="#000000",
        )
    span_w = wacc_max - wacc_min
    axes[0].text(0.00, 1.04, f"a  WACC sweep (span {span_w:+.0f} pp)",
                 transform=axes[0].transAxes, ha="left", va="bottom",
                 fontsize=8, fontweight="bold")
    axes[0].set_ylabel("Grid-cost margin (%)")
    axes[0].set_xlabel("WACC")
    axes[0].axhline(0, **ZERO_LINE_KW)
    axes[0].grid(**GRID_KW)
    axes[0].set_axisbelow(True)

    # Right: CAPEX sweep (FOAK / ATB_Mid / NOAK)
    labels_capex = [s.replace("ATB_Mid", "ATB-Mid") for s, _ in capex_pts]
    prems_capex = [p for _, p in capex_pts]
    colors_capex = [
        MAIN_FILLS[0] if p > prems_capex[1] else
        (MAIN_FILLS[1] if p < prems_capex[1] else MAIN_FILLS[3])
        for p in prems_capex
    ]
    bars_r = axes[1].bar(
        labels_capex, prems_capex,
        color=colors_capex, edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=0.6,
    )
    for b, v in zip(bars_r, prems_capex):
        axes[1].text(
            b.get_x() + b.get_width() / 2,
            v + (8 if v < 0 else -8),
            f"{v:.0f}%",
            ha="center", va="center", fontsize=8, color="#000000",
        )
    span_c = capex_max - capex_min
    axes[1].text(0.00, 1.04, f"b  CAPEX learning (span {span_c:+.0f} pp)",
                 transform=axes[1].transAxes, ha="left", va="bottom",
                 fontsize=8, fontweight="bold")
    axes[1].set_xlabel("SMR CAPEX scenario")
    axes[1].axhline(0, **ZERO_LINE_KW)
    axes[1].grid(**GRID_KW)
    axes[1].set_axisbelow(True)

    # Sync y-limits so the comparison is fair
    ymin = min(min(prems_wacc), min(prems_capex))
    ymax = max(max(prems_wacc), max(prems_capex))
    pad = 0.06 * (ymax - ymin)
    for ax in axes:
        ax.set_ylim(ymin - pad, ymax + pad)

    # Caption hook: which lever wins?
    ratio = span_c / span_w if span_w != 0 else float("inf")
    note = (
        f"CAPEX learning provides {ratio:.1f}× the Premium leverage of WACC reduction"
        if ratio > 1.0 else
        f"WACC reduction provides {1.0 / ratio:.1f}× the Premium leverage of CAPEX learning"
    )
    fig.text(
        0.5, -0.01, note, ha="center", fontsize=8, color="#000000",
        style="italic",
    )

    fig.tight_layout(rect=(0.0, 0.06, 1.0, 0.98))
    save_triplet(fig, "fig11_s7_wacc_leverage", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig11_s7_wacc_leverage)

In [ ]:
# Unused figure (not in current main.pdf): fig_sensitivity_tornado
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig_sensitivity_tornado() -> None:
    def hrp_percent(run_id: str) -> float:
        return float(df.loc[df.run_id == run_id, "heat_recovery_premium"].iloc[0] * 100.0)

    baseline = hrp_percent("case2_ATB_Mid")
    axes_def = [
        ("SMR CAPEX", "case2_FOAK", "case2_NOAK"),
        ("Market year", "case2_year2024", "case2_year2022"),
        ("WACC", "case2_wacc_100", "case2_wacc_50"),
        ("Carbon price", "case2_co2_0", "case2_co2_100"),
        ("DC size", "case2_load_x050", "case2_load_x300"),
        ("Absorption CAPEX", "case2_smr_ATB_Mid_abs_Turnkey_High", "case2_smr_ATB_Mid_abs_Bare_Low"),
        ("PUE", "case2_pue150", "case2_pue110"),
        ("BESS on/off", "case2_bess_off", "case2_bess_on"),
    ]

    axis_spans = []
    for label, run_a, run_b in axes_def:
        va = hrp_percent(run_a)
        vb = hrp_percent(run_b)
        lo, hi = min(va, vb), max(va, vb)
        axis_spans.append((label, lo, hi, hi - lo))
    axis_spans.sort(key=lambda row: row[3], reverse=True)

    labels = [row[0] for row in axis_spans]
    lows = [row[1] for row in axis_spans]
    highs = [row[2] for row in axis_spans]
    spans = [row[3] for row in axis_spans]
    ypos = np.arange(len(axis_spans))[::-1]

    fig, ax = plt.subplots(figsize=(BODY_W, 3.8))
    ax.axvline(baseline, color="#333333", lw=1.0, ls="--", zorder=1)
    ax.annotate(
        f"ATB-Mid baseline ({baseline:.0f}%)",
        xy=(baseline, len(axis_spans) - 0.55), xytext=(POINT_LABEL_OFFSET_PT, 0),
        textcoords="offset points",
        ha="left", va="center", fontsize=8, color="#333333",
    )

    worse_color = MAIN_FILLS[1]
    better_color = MAIN_FILLS[0]
    for y, lo, hi in zip(ypos, lows, highs):
        left = min(lo, baseline)
        right = max(hi, baseline)
        if left < baseline:
            ax.barh(y, baseline - left, left=left, height=0.62,
                    color=worse_color, edgecolor="white", linewidth=0.5, zorder=2)
        if right > baseline:
            ax.barh(y, right - baseline, left=baseline, height=0.62,
                    color=better_color, edgecolor="white", linewidth=0.5, zorder=2)
        annotate_horizontal_endpoint(
            ax, lo, y, f"{lo:.0f}%", side="left", fontsize=8, color=MAIN_STROKES[1]
        )
        annotate_horizontal_endpoint(
            ax, hi, y, f"{hi:.0f}%", side="right", fontsize=8, color=MAIN_STROKES[0]
        )

    ax.set_yticks(ypos)
    ax.set_yticklabels([f"{label}\n(span {span:.0f} pp)" for label, span in zip(labels, spans)])
    ax.set_xlabel("Case 2 grid-cost margin (%)")
    ax.set_xlim(min(lows) - 90, max(highs) + 40)
    ax.set_ylim(-1.0, len(axis_spans) - 0.4)
    ax.set_axisbelow(True)
    ax.grid(axis="x", alpha=0.25, linestyle="--", linewidth=0.5)
    ax.tick_params(direction="out", length=3)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    handles = [
        Patch(facecolor=worse_color, edgecolor="white", label="more negative than baseline"),
        Patch(facecolor=better_color, edgecolor="white", label="less negative than baseline"),
    ]
    ax.legend(handles=handles, loc="lower left", frameon=False, fontsize=8,
              bbox_to_anchor=(0.0, 0.02))

    fig.tight_layout()
    save_triplet(fig, "fig_sensitivity_tornado", str(FIGURES))
    display(fig)
    plt.close(fig)


maybe_render_unused_figure(fig_sensitivity_tornado)


In [ ]:
# Unused figure (not in current main.pdf): fig_si_water_3tier
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def fig_si_water_3tier() -> None:
    base = df[df.group == "main_baseline"].sort_values("case_id").reset_index(drop=True)
    cases = base.case_id.astype(int).values
    labels = [f"C{c}" for c in cases]

    direct = base.water_direct_site_l_per_mwh_e.values
    indirect = base.water_indirect_generation_l_per_mwh_e.values
    total = base.water_total_l_per_mwh_e.values
    scarcity = base.water_scarcity_m3_world_eq_per_mwh_e.values

    fig, axes = plt.subplots(1, 2, figsize=(BODY_W, 3.0),
                              gridspec_kw={"width_ratios": [1.0, 1.0]})

    # Left: stacked direct + indirect (linear scale; uses two colors)
    axes[0].bar(labels, direct, color=MAIN_FILLS[0],
                edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=0.62,
                label="Direct site (DC cooling tower)")
    axes[0].bar(labels, indirect, bottom=direct, color=MAIN_FILLS[1],
                edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=0.62,
                label="Indirect generation (Macknick)")
    for i, (d, t) in enumerate(zip(direct, total)):
        axes[0].text(i, t + max(total) * 0.025,
                     f"{t:,.0f}", ha="center", va="bottom",
                     fontsize=8, color="#000000")
        if d > 1.0:
            axes[0].text(i, d / 2, f"{d:.0f}", ha="center", va="center",
                         fontsize=8, color="#FFFFFF")
    axes[0].set_ylabel("Water (L/MWh$_\\mathrm{e\\,IT}$)")
    axes[0].set_axisbelow(True)
    axes[0].grid(**GRID_KW)
    axes[0].legend(loc="upper center", fontsize=8, frameon=False,
                    ncol=1, bbox_to_anchor=(0.5, -0.10))
    add_panel_label(axes[0], "a")

    # Right: scarcity-weighted (m3 world-eq / MWh_e_IT)
    bars = axes[1].bar(labels, scarcity, color=[CASE_FILL[c] for c in cases],
                        edgecolor=BLOCK_EDGE, linewidth=EDGE_LW, width=0.62)
    for b, v in zip(bars, scarcity):
        axes[1].text(b.get_x() + b.get_width() / 2,
                     v + max(scarcity) * 0.025,
                     f"{v:.2f}", ha="center", va="bottom",
                     fontsize=8, color="#000000")
    axes[1].set_ylabel("Scarcity-weighted\n(m$^3$ world-eq/MWh$_\\mathrm{e\\,IT}$)")
    axes[1].set_axisbelow(True)
    axes[1].grid(**GRID_KW)
    add_panel_label(axes[1], "b")

    # Below the figure, surface the absorption-vs-VCC reversal as caption hook
    delta_direct = direct[2] - direct[1]   # Case 2 - Case 1 (direct site)
    delta_indirect = indirect[2] - indirect[1]
    note = (
        f"Case 2 vs Case 1: direct site {delta_direct:+.1f}, "
        f"indirect generation {delta_indirect:+.0f} L/MWh$_\\mathrm{{e\\,IT}}$ — "
        f"absorption chiller trades indirect for direct water"
    )
    fig.text(0.5, -0.06, note, ha="center", fontsize=8, color="#000000",
             style="italic")

    fig.tight_layout()
    save_triplet(fig, "fig_si_water_3tier", str(FIGURES))
    display(fig)
    plt.close(fig)

maybe_render_unused_figure(fig_si_water_3tier)

In [ ]:
# Unused figure (not in current main.pdf): graphical_abstract
# Kept after manuscript Figure 1-Figure 10 cells for supplementary or exploratory reuse.
def graphical_abstract() -> None:
    apply_sci_style("poster")

    # ---- Left panel data: S5 SMR×absorption viability heatmap ----------
    s5 = df[df.group == "s5_feasibility_2d"].copy()
    smr_order = ["NOAK", "Low_Mid", "ATB_Mid", "High_Mid", "FOAK"]
    abs_order = ["Bare_Low", "Mid_Low", "Baseline", "Mid_High", "Turnkey_High"]
    smr_vals = [2250, 5000, 7615, 11000, 14700]
    abs_vals = [450, 600, 750, 900, 1200]
    piv = s5.pivot_table(index="smr_capex_tag", columns="absorption_capex_tag",
                         values="premium_pct").reindex(smr_order)[abs_order]
    data = piv.values

    cmap = LinearSegmentedColormap.from_list(
        "sf_div",
        [PALETTE["fill_salmon"], "#FFFFFF", PALETTE["fill_blue"]],
        N=256,
    )
    vmax = max(abs(data.min()), abs(data.max()), 30)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

    # ---- Right panel data: S6 carbon-price crossover -------------------
    s6 = df[df.group == "s6_carbon_price"].copy().sort_values(
        ["case_id", "carbon_price_usd_per_tco2"]
    )
    x_extrap = np.linspace(0, 250, 256)
    case_lines: dict[int, tuple[np.ndarray, np.ndarray, np.ndarray]] = {}
    case0_at_x: tuple[float, float] | None = None
    crossings: dict[int, float] = {}
    for cid in (0, 1, 2, 3):
        sub = s6[s6.case_id == cid]
        x = sub.carbon_price_usd_per_tco2.values
        y = sub.tac_usd_per_yr.values / 1e6
        slope = (y[-1] - y[0]) / (x[-1] - x[0])
        intercept = y[0]
        case_lines[cid] = (x_extrap, intercept + slope * x_extrap, x, y)
        if cid == 0:
            case0_at_x = (intercept, slope)
        else:
            c0_int, c0_slope = case0_at_x
            denom = (slope - c0_slope)
            if abs(denom) > 1e-9:
                p_cross = (c0_int - intercept) / denom
                if 0 < p_cross < 260:
                    crossings[cid] = p_cross

    # ---- Compose -----------------------------------------------------------
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.0, 5.6),
                                    gridspec_kw={"width_ratios": [1.0, 1.0]})

    # Left: S5 heatmap
    im = axL.imshow(data, cmap=cmap, norm=norm, aspect="auto")
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            axL.text(j, i, f"{v:+.0f}%", ha="center", va="center", fontsize=12,
                     color="white" if abs(v) > 0.6 * vmax else "#000000")
    axL.contour(
        np.arange(data.shape[1]),
        np.arange(data.shape[0]),
        data,
        levels=[0.0],
        colors=[PALETTE["stroke_navy"]],
        linewidths=2.0,
        linestyles="--",
    )
    axL.set_xticks(np.arange(len(abs_order)))
    axL.set_xticklabels([f"${v}" for v in abs_vals], fontsize=11)
    axL.set_yticks(np.arange(len(smr_order)))
    axL.set_yticklabels([f"${v:,}" for v in smr_vals], fontsize=11)
    axL.set_xlabel(r"Absorption CAPEX (\$/kW$_\mathrm{c}$)", fontsize=13)
    axL.set_ylabel(r"SMR CAPEX (\$/kW$_\mathrm{e}$)", fontsize=13)
    axL.set_xticks(np.arange(-0.5, len(abs_order), 1), minor=True)
    axL.set_yticks(np.arange(-0.5, len(smr_order), 1), minor=True)
    axL.grid(which="minor", color="white", linewidth=2.0)
    axL.tick_params(which="minor", length=0)
    add_panel_label(axL, "a", x=-0.16, y=1.04, fontsize=15)
    cbar = fig.colorbar(im, ax=axL, fraction=0.046, pad=0.04)
    cbar.set_label("Grid-cost margin (%)", fontsize=12)

    # Right: S6 carbon-price crossover
    for cid in (0, 1, 2, 3):
        x_e, y_e, x_pts, y_pts = case_lines[cid]
        axR.plot(x_e, y_e,
                 color=CASE_LINE[cid], linewidth=2.0, zorder=2,
                 label=f"C{cid}")
        axR.scatter(x_pts, y_pts, marker=CASE_MARKER[cid], s=80,
                    facecolor=CASE_FILL[cid], edgecolor=CASE_LINE[cid],
                    linewidth=1.0, zorder=4)
    # Group near-identical Case 1 / Case 2 crossovers — same rule as Fig 10.
    cid_groups: list[tuple[list[int], float]] = []
    used: set[int] = set()
    for cid in (1, 2, 3):
        if cid not in crossings or cid in used:
            continue
        partners = [cid]
        used.add(cid)
        for other in (1, 2, 3):
            if other in crossings and other not in used and abs(
                crossings[other] - crossings[cid]
            ) < 3.0:
                partners.append(other)
                used.add(other)
        cid_groups.append(
            (sorted(partners), float(np.mean([crossings[c] for c in partners])))
        )

    for cids, p_cross in cid_groups:
        c0_int, c0_slope = case0_at_x
        tac_cross = c0_int + c0_slope * p_cross
        primary = cids[-1]
        axR.axvline(p_cross, color=CASE_LINE[primary],
                    linestyle=":", linewidth=1.4, alpha=0.85, zorder=1)
        label = "/".join(f"C{c}" for c in cids) + f": \\${p_cross:.0f}"
        axR.annotate(
            label,
            xy=(p_cross, tac_cross),
            xytext=(10, 12 if 2 in cids else -22),
            textcoords="offset points",
            fontsize=11, color=CASE_LINE[primary], ha="left",
        )
    if crossings:
        p_max = max(crossings.values())
        axR.axvspan(p_max, 250, alpha=0.07,
                    color=PALETTE["fill_blue"], zorder=0)
        x_norm = ((p_max + 250) / 2) / 250
        axR.text(x_norm, 0.965, "Nuclear $<$ grid",
                 transform=axR.transAxes, ha="center", va="top",
                 fontsize=11, color=MAIN_STROKES[0], style="italic")

    axR.set_xlim(0, 250)
    axR.set_xlabel(r"Carbon price (\$/tCO$_2$)", fontsize=13)
    axR.set_ylabel(r"TAC (M\$/yr)", fontsize=13)
    axR.set_axisbelow(True)
    axR.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.6)
    axR.legend(loc="upper left", frameon=False, fontsize=11,
               handlelength=1.4, ncol=2)
    add_panel_label(axR, "b", x=-0.16, y=1.04, fontsize=15)

    fig.tight_layout()
    save_triplet(fig, "graphical_abstract", str(FIGURES))
    display(fig)
    plt.close(fig)
    apply_manuscript_style("ae_single")  # restore manuscript style for any later figs

maybe_render_unused_figure(graphical_abstract)